
# ECON 662 — Homework 2

## Combined Notebook

This notebook combines all sections of the project:

- Part (a): NFXP
- Part (b): Hotz-Miller inversion
- Part (c.1): Forward simulation using Equation (5)
- Part (c.2): Forward simulation using Equations (6) and (7)
- Part (d): Scott (2014) linear regression

Run the notebook from top to bottom to reproduce each section's results and the final comparison table.


In [1]:

master_results = {}
master_common = {}

print("Initialized master_results and master_common.")


Initialized master_results and master_common.


---
## Part (a): Nested Fixed Point


# ECON 662 — Problem Set 2
## NFXP Estimation

### Import Libraries 


In [2]:
import time
import numpy as np
import pandas as pd
from scipy.stats import norm        # norm.logpdf — only for OLS sigma check
from scipy.optimize import minimize # outer Nelder-Mead only

# No other packages. Everything else is plain numpy matrix algebra.

###  Load Data


In [3]:
# Columns: i (bus id), t (month 1-100), a (action: 0=maintain, 1=replace),
#          x (mileage state: 1-7), rc (replacement cost, continuous)
df = pd.read_csv("ddc_pset.csv")
df = df.sort_values(["i", "t"]).reset_index(drop=True)

print(f"Loaded {len(df):,} observations for {df['i'].nunique():,} buses.")
print(f"RC range in data: [{df['rc'].min():.4f}, {df['rc'].max():.4f}]")
print(f"Replacement rate: {df['a'].mean():.4f}  ({df['a'].sum():,} replacements)")
print()
df.head(10)

Loaded 100,000 observations for 1,000 buses.
RC range in data: [31.5000, 62.5000]
Replacement rate: 0.1724  (17,241 replacements)



,i,t,a,x,rc
0,1,1,0,4,45.0
1,1,2,0,5,49.0
2,1,3,0,6,54.0
3,1,4,1,7,47.0
4,1,5,0,1,56.0
5,1,6,0,2,52.5
6,1,7,0,3,55.0
7,1,8,0,4,50.5
8,1,9,0,5,52.0
9,1,10,0,6,42.0


### Global Constants

In [4]:
BETA         = 0.95          # discount factor (given in problem)
N_X          = 7             # number of mileage states {1, 2, ..., 7}
N_RC_BINS    = 8             # number of equal-width RC bins (professor uses 8)
GAMMA_EULER  = 0.5772156649  # Euler-Mascheroni constant (mean of Type I EV dist.)

print(f"beta        = {BETA}")
print(f"N_X         = {N_X}  (mileage states)")
print(f"N_RC_BINS   = {N_RC_BINS}  (empirical RC bins)")
print(f"State space = {N_X} x {N_RC_BINS} = {N_X * N_RC_BINS} total states")

beta        = 0.95
N_X         = 7  (mileage states)
N_RC_BINS   = 8  (empirical RC bins)
State space = 7 x 8 = 56 total states


### Create Equal-Width RC Bins from the Data

Instead of using the AR(1) theory to build a grid, we simply cut the observed RC range into `N_RC_BINS` equal-width intervals and represent each bin by its midpoint.

> "From now on, rc means the grid mean instead of the actual rc value." 
> Similar to Professor's approach

This fixes the grid once from the data before any estimation begins.

In [5]:
# ── (a) Bin edges: N_RC_BINS+1 evenly spaced points over [RC_min, RC_max] ──
rc_min = df['rc'].min()
rc_max = df['rc'].max()

bin_edges = np.linspace(rc_min, rc_max, N_RC_BINS + 1)   # shape (9,)
print("Bin edges:")
print(np.round(bin_edges, 4))

# ── (b) Bin midpoints: the 'representative RC value' for each bin ──
bin_midpoints = (bin_edges[:-1] + bin_edges[1:]) / 2.0   # shape (8,)
print("\nBin midpoints (rc_grid):")
print(np.round(bin_midpoints, 4))

# ── (c) Assign each observation to its bin ──
# np.searchsorted finds the bin index: bin_edges[j] <= rc < bin_edges[j+1]
# Subtract 1 to make 0-indexed; clip to [0, N_RC_BINS-1] for edge cases
rc_bin_idx = np.searchsorted(bin_edges[1:], df['rc'].values, side='left')
rc_bin_idx = np.clip(rc_bin_idx, 0, N_RC_BINS - 1)       # shape (100000,)

df['rc_bin'] = rc_bin_idx                                 # which bin (0-indexed)
df['rc_mid'] = bin_midpoints[rc_bin_idx]                  # bin representative value

print("\nData after binning (first 10 rows):")
print(df[['i','t','a','x','rc','rc_bin','rc_mid']].head(10))

# ── (d) Observations per bin ──
print("\nObservations per RC bin:")
bin_counts = np.bincount(rc_bin_idx, minlength=N_RC_BINS)
for j in range(N_RC_BINS):
    print(f"  Bin {j}  [{bin_edges[j]:.2f}, {bin_edges[j+1]:.2f}]  "
          f"midpoint={bin_midpoints[j]:.2f}  n={bin_counts[j]:,}")

Bin edges:
[31.5   35.375 39.25  43.125 47.    50.875 54.75  58.625 62.5  ]

Bin midpoints (rc_grid):
[33.4375 37.3125 41.1875 45.0625 48.9375 52.8125 56.6875 60.5625]

Data after binning (first 10 rows):
   i   t  a  x    rc  rc_bin   rc_mid
0  1   1  0  4  45.0       3  45.0625
1  1   2  0  5  49.0       4  48.9375
2  1   3  0  6  54.0       5  52.8125
3  1   4  1  7  47.0       3  45.0625
4  1   5  0  1  56.0       6  56.6875
5  1   6  0  2  52.5       5  52.8125
6  1   7  0  3  55.0       6  56.6875
7  1   8  0  4  50.5       4  48.9375
8  1   9  0  5  52.0       5  52.8125
9  1  10  0  6  42.0       2  41.1875

Observations per RC bin:
  Bin 0  [31.50, 35.38]  midpoint=33.44  n=3,000
  Bin 1  [35.38, 39.25]  midpoint=37.31  n=5,000
  Bin 2  [39.25, 43.12]  midpoint=41.19  n=15,000
  Bin 3  [43.12, 47.00]  midpoint=45.06  n=24,000
  Bin 4  [47.00, 50.88]  midpoint=48.94  n=26,000
  Bin 5  [50.88, 54.75]  midpoint=52.81  n=18,000
  Bin 6  [54.75, 58.62]  midpoint=56.69  n=7,000
  Bi

### Build the Empirical Transition Matrix $\Pi$

We count how many times in the data $RC$ moves from bin $j$ to bin $j'$ in consecutive periods:

$$C_{j,j'} = \#\{t : RC_{t-1} \in \text{bin}\, j,\ RC_t \in \text{bin}\, j'\}$$

Then normalize each row to get probabilities:

$$\Pi_{j,j'} = \frac{C_{j,j'}}{\sum_{j''} C_{j,j''}}$$

This matrix $\Pi$ is fixed for the entire estimation, it never changes during optimization.

In [6]:
# (a) Extract consecutive (bin_{t-1}, bin_t) pairs within each bus ──
bin_prev_list, bin_curr_list = [], []

for _, g in df.groupby("i"):
    bins = g['rc_bin'].values
    bin_prev_list.append(bins[:-1])   # bin of RC_{t-1}
    bin_curr_list.append(bins[1:])    # bin of RC_t

bin_prev = np.concatenate(bin_prev_list)  # shape (99000,)
bin_curr = np.concatenate(bin_curr_list)  # shape (99000,)

print(f"Transition pairs extracted: {len(bin_prev):,}")

#  (b) Count matrix C: C[j, j'] = # transitions from bin j to bin j' ──
C = np.zeros((N_RC_BINS, N_RC_BINS), dtype=float)

for j_from, j_to in zip(bin_prev, bin_curr):
    C[j_from, j_to] += 1

print(f"\nCount matrix C  ({N_RC_BINS} x {N_RC_BINS}):")
print("  (C[j, j'] = # times RC moved from bin j to bin j')")
print(C.astype(int))
print(f"\nRow totals (# times RC was in each bin):")
print(C.sum(axis=1).astype(int))

# (c) Normalize rows → transition probability matrix Pi ──
row_totals = C.sum(axis=1, keepdims=True)   # shape (8, 1)
Pi = C / row_totals                          # shape (8, 8)

print(f"\nTransition matrix Pi  ({N_RC_BINS} x {N_RC_BINS}):")
print("  (Pi[j, j'] = P(RC moves to bin j' | currently in bin j))")
print(np.round(Pi, 4))

#  (d) Sanity check: rows must sum to 1 ──
row_sums = Pi.sum(axis=1)
print(f"\nRow sums (all must equal 1.0):")
print(np.round(row_sums, 10))

Transition pairs extracted: 99,000

Count matrix C  (8 x 8):
  (C[j, j'] = # times RC moved from bin j to bin j')
[[   0 1000 2000    0    0    0    0    0]
 [1000 2000 1000 1000    0    0    0    0]
 [2000    0 6000 3000 3000 1000    0    0]
 [   0 2000 2000 7000 9000 2000 1000    0]
 [   0    0 3000 7000 7000 8000 1000    0]
 [   0    0 1000 5000 5000 2000 4000 1000]
 [   0    0    0    0 2000 4000    0 1000]
 [   0    0    0    0    0 1000 1000    0]]

Row totals (# times RC was in each bin):
[ 3000  5000 15000 23000 26000 18000  7000  2000]

Transition matrix Pi  (8 x 8):
  (Pi[j, j'] = P(RC moves to bin j' | currently in bin j))
[[0.     0.3333 0.6667 0.     0.     0.     0.     0.    ]
 [0.2    0.4    0.2    0.2    0.     0.     0.     0.    ]
 [0.1333 0.     0.4    0.2    0.2    0.0667 0.     0.    ]
 [0.     0.087  0.087  0.3043 0.3913 0.087  0.0435 0.    ]
 [0.     0.     0.1154 0.2692 0.2692 0.3077 0.0385 0.    ]
 [0.     0.     0.0556 0.2778 0.2778 0.1111 0.2222 0.0556]
 [0.

We now have an 8 × 8 matrix $\Pi$ built entirely from data

### OLS for AR(1) Parameters

We estimate $(\rho_0, \rho_1, \sigma_\rho)$ once via OLS before the optimizer runs.
The AR(1) likelihood is separable from the choice likelihood; therefore, we can maximise each part independently.

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

where $\mathbf{X} = [\mathbf{1} \mid RC_{t-1}]$ and $\mathbf{y} = RC_t$.

In [7]:
# (a) Build (RC_{t-1}, RC_t) pairs using actual (continuous) RC values 
rc_prev_list, rc_curr_list = [], []
for _, g in df.groupby("i"):
    rc_vals = g['rc'].values
    rc_prev_list.append(rc_vals[:-1])
    rc_curr_list.append(rc_vals[1:])

rc_prev_vec = np.concatenate(rc_prev_list)   # shape (99000,)
rc_curr_vec = np.concatenate(rc_curr_list)   # shape (99000,)

# (b) Design matrix X = [1, RC_{t-1}], shape (99000, 2) 
X = np.column_stack([np.ones(len(rc_prev_vec)), rc_prev_vec])

print(f"Design matrix X shape: {X.shape}")
print("First 5 rows of X:")
print(X[:5])

# (c) OLS normal equations: β = (X'X)^{-1} X'y 
XtX = X.T @ X          # (2, 2)
Xty = X.T @ rc_curr_vec  # (2,)

print(f"\nX'X =\n{XtX}")
print(f"\nX'y = {Xty}")

beta_ols = np.linalg.solve(XtX, Xty)
rho0_ols, rho1_ols = beta_ols

resid_ols    = rc_curr_vec - X @ beta_ols
sigma_rho_ols = resid_ols.std(ddof=1)

print("\n--- OLS AR(1) estimates (fixed for entire estimation) ---")
print(f"  rho0      = {rho0_ols:.4f}")
print(f"  rho1      = {rho1_ols:.4f}")
print(f"  sigma_rho = {sigma_rho_ols:.4f}")
#print("\nThese match Table 1 in the professor's solution: rho0=17.8269, rho1=0.6249, sigma=4.606")

Design matrix X shape: (99000, 2)
First 5 rows of X:
[[ 1. 45.]
 [ 1. 49.]
 [ 1. 54.]
 [ 1. 47.]
 [ 1. 56.]]

X'X =
[[9.9000000e+04 4.7025000e+06]
 [4.7025000e+06 2.2682125e+08]]

X'y = [4.7035000e+06 2.2557375e+08]

--- OLS AR(1) estimates (fixed for entire estimation) ---
  rho0      = 17.8269
  rho1      = 0.6249
  sigma_rho = 4.6060


### State-Level Aggregation

From the professor notes: "Notice that all the estimations are done at state level"

There are $7 \times 8 = 56$ states. For each state $(x, j)$ we record:
- $N_1(x, j)$ = number of observations where action = 1 (replace)
- $N_0(x, j)$ = number of observations where action = 0 (maintain)

The likelihood then sums over states (56 terms), not observations (100,000 terms).
#Same mathematical result, much faster computation.

In [8]:
#  Count N_0 and N_1 for each state (x, rc_bin) 
# State matrix shape: (N_X, N_RC_BINS)
# x is 1-indexed in data, rc_bin is 0-indexed

N1 = np.zeros((N_X, N_RC_BINS))   # N1[xi, j] = # replace decisions in state (xi+1, j)
N0 = np.zeros((N_X, N_RC_BINS))   # N0[xi, j] = # maintain decisions in state (xi+1, j)

x_data   = df['x'].values.astype(int)        # 1-indexed
bin_data = df['rc_bin'].values.astype(int)   # 0-indexed
a_data   = df['a'].values.astype(int)

for xi_obs, j_obs, a_obs in zip(x_data, bin_data, a_data):
    if a_obs == 1:
        N1[xi_obs - 1, j_obs] += 1   # subtract 1 for 0-indexing
    else:
        N0[xi_obs - 1, j_obs] += 1

print(f"N1 shape: {N1.shape}  (rows=mileage x, cols=RC bin)")
print(f"\nN1  — # REPLACE decisions per state (x, rc_bin):")
print("        ", [f"bin{j}" for j in range(N_RC_BINS)])
for xi in range(N_X):
    print(f"  x={xi+1}:  {N1[xi].astype(int)}")

print(f"\nN0  — # MAINTAIN decisions per state (x, rc_bin):")
for xi in range(N_X):
    print(f"  x={xi+1}:  {N0[xi].astype(int)}")

print(f"\nTotal observations: N0+N1 = {(N0+N1).sum():.0f}  (should be 100,000)")

#  Empirical replacement rate per state 
N_total = N0 + N1
P_emp = np.where(N_total > 0, N1 / N_total, np.nan)  # empirical CCP at each state
print("\nEmpirical P(replace) per state (raw frequency):")
print("        ", [f"bin{j}" for j in range(N_RC_BINS)])
for xi in range(N_X):
    print(f"  x={xi+1}:  {np.round(P_emp[xi], 3)}")

N1 shape: (7, 8)  (rows=mileage x, cols=RC bin)

N1  — # REPLACE decisions per state (x, rc_bin):
         ['bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5', 'bin6', 'bin7']
  x=1:  [20 23 29 22 15  7  0  0]
  x=2:  [42 67 94 89 65 20  4  2]
  x=3:  [172 158 339 302 220 107  15   3]
  x=4:  [187 237 615 693 533 225  63   9]
  x=5:  [ 200  274  788 1044  876  447  143   20]
  x=6:  [  91  221  642 1039  844  647  144   22]
  x=7:  [ 185  248  687 1369 1433 1094  323   83]

N0  — # MAINTAIN decisions per state (x, rc_bin):
  x=1:  [ 682 1164 3091 4075 4092 2800 1023  237]
  x=2:  [ 499  942 3007 4043 4373 2616 1045  236]
  x=3:  [ 477  753 2183 3728 4315 2615 1109  270]
  x=4:  [ 239  454 1604 3365 3619 2395  880  357]
  x=5:  [ 122  242 1045 2011 2604 1940  925  223]
  x=6:  [  45  130  495 1166 1443 1424  497  221]
  x=7:  [  39   87  381 1054 1568 1663  829  317]

Total observations: N0+N1 = 100000  (should be 100,000)

Empirical P(replace) per state (raw frequency):
         ['bin0', '

### Bellman Equation 

$$v_0(x,j) = -\theta_1 x + \beta \underbrace{\sum_{j'} \Pi_{j,j'} EV(\min(x{+}1,7),\ j')}_{\text{row } j \text{ of } EV \cdot \Pi^\top}$$

$$v_1(x,j) = -\theta_2\, rc_j + \beta \underbrace{\sum_{j'} \Pi_{j,j'} EV(1,\ j')}_{\text{row } j \text{ of } EV \cdot \Pi^\top, \text{ first row only}}$$

$$EV(x,j) = \log(e^{v_0} + e^{v_1}) + \gamma_E$$


In [9]:
def solve_bellman(theta1, theta2, rc_grid, Pi, tol=1e-8, maxiter=2000):
    """
    Value function iteration.
    Pi is now 8x8 and fixed; rc_grid is the 8 bin midpoints.

    Returns
    -------
    EV      : ndarray (N_X, N_RC_BINS)  — integrated value function
    v0_grid : ndarray (N_X, N_RC_BINS)  — value of maintaining
    v1_grid : ndarray (N_X, N_RC_BINS)  — value of replacing
    """
    N_rc = len(rc_grid)
    EV   = np.zeros((N_X, N_rc))   # start from zero

    # Pre-compute objects that never change across iterations
    x_idx    = np.arange(N_X)                    # [0, 1, 2, 3, 4, 5, 6]
    x_next_0 = np.minimum(x_idx + 1, N_X - 1)   # maintain: x+1, capped at 6 (state 7)
    x_next_1 = 0                                  # replace: always reset to state 1

    # Flow utility of maintenance: -theta1 * x, shape (N_X, 1)
    u0_base = (-theta1 * (x_idx + 1)).reshape(N_X, 1)

    # Flow utility of replacement: -theta2 * rc_grid[j], repeated across mileage states
    u1 = np.broadcast_to((-theta2 * rc_grid).reshape(1, N_rc), (N_X, N_rc))

    for iteration in range(maxiter):

        # ── Continuation values: one matrix multiply 
        # EV_cont[xi, j] = sum_{j'} Pi[j, j'] * EV[xi, j']
        #                 = (EV @ Pi.T)[xi, j]
        # Shape: (N_X, N_rc) = (7, 8) @ (8, 8) transposed... wait:
        # EV is (7, 8), Pi.T is (8, 8), so EV @ Pi.T is (7, 8)  
        EV_cont = EV @ Pi.T                              # (7, 8)

        # ── Choice-specific value functions 
        v0 = u0_base + BETA * EV_cont[x_next_0, :]      # (7, 8)
        v1 = u1 + np.broadcast_to(EV_cont[x_next_1, :], (N_X, N_rc)) * BETA

        # ── Log-sum-exp (numerically stable) 
        vmax   = np.maximum(v0, v1)
        EV_new = np.log(np.exp(v0 - vmax) + np.exp(v1 - vmax)) + vmax + GAMMA_EULER

        diff = np.max(np.abs(EV_new - EV))
        EV   = EV_new

        if diff < tol:
            break

    return EV, v0, v1

#### Inspect the Bellman output at initial guess $\theta_1 = \theta_2 = 1$

In [10]:
rc_grid = bin_midpoints   # the 8 bin midpoints — our RC grid

EV_init, v0_init, v1_init = solve_bellman(1.0, 1.0, rc_grid, Pi)

print(f"rc_grid (bin midpoints): {np.round(rc_grid, 2)}")
print(f"\nEV shape: {EV_init.shape}  (7 mileage states × 8 RC bins)")

print("\n── EV (integrated value function) ──")
print(f"{'':6}", "  ".join(f"bin{j}" for j in range(N_RC_BINS)))
for xi in range(N_X):
    print(f"  x={xi+1}:  {np.round(EV_init[xi], 2)}")

print("\n── v0_grid (value of MAINTAINING) ──")
print(f"{'':6}", "  ".join(f"bin{j}" for j in range(N_RC_BINS)))
for xi in range(N_X):
    print(f"  x={xi+1}:  {np.round(v0_init[xi], 2)}")

print("\n── v1_grid (value of REPLACING) ──")
print(f"{'':6}", "  ".join(f"bin{j}" for j in range(N_RC_BINS)))
for xi in range(N_X):
    print(f"  x={xi+1}:  {np.round(v1_init[xi], 2)}")

print("\n── v1 - v0  (positive => prefer replace) ──")
for xi in range(N_X):
    print(f"  x={xi+1}:  {np.round(v1_init[xi] - v0_init[xi], 3)}")

rc_grid (bin midpoints): [33.44 37.31 41.19 45.06 48.94 52.81 56.69 60.56]

EV shape: (7, 8)  (7 mileage states × 8 RC bins)

── EV (integrated value function) ──
       bin0  bin1  bin2  bin3  bin4  bin5  bin6  bin7
  x=1:  [-109.12 -109.12 -109.12 -109.12 -109.12 -109.12 -109.12 -109.12]
  x=2:  [-114.42 -114.42 -114.42 -114.42 -114.42 -114.42 -114.42 -114.42]
  x=3:  [-118.94 -118.94 -118.94 -118.94 -118.94 -118.94 -118.94 -118.94]
  x=4:  [-122.65 -122.65 -122.65 -122.65 -122.65 -122.65 -122.65 -122.65]
  x=5:  [-125.51 -125.51 -125.51 -125.51 -125.51 -125.51 -125.51 -125.51]
  x=6:  [-127.46 -127.46 -127.46 -127.46 -127.46 -127.46 -127.46 -127.46]
  x=7:  [-128.46 -128.46 -128.46 -128.46 -128.46 -128.46 -128.46 -128.46]

── v0_grid (value of MAINTAINING) ──
       bin0  bin1  bin2  bin3  bin4  bin5  bin6  bin7
  x=1:  [-109.7 -109.7 -109.7 -109.7 -109.7 -109.7 -109.7 -109.7]
  x=2:  [-115. -115. -115. -115. -115. -115. -115. -115.]
  x=3:  [-119.52 -119.52 -119.52 -119.52 -119.52 

### Choice Probabilities

Here, we already replaced every observed RC with its bin midpoint.
So each observation maps to exactly one grid index, just a direct array lookup:

$$P(\text{replace} \mid x_i, \text{bin}_i) = \frac{1}{1 + \exp(v_0[x_i,\, \text{bin}_i] - v_1[x_i,\, \text{bin}_i])}$$

Vectorised over all 100,000 observations in one line, or, equivalently, over all 56 states.

In [11]:
def compute_choice_probs_grid(v0_grid, v1_grid):
    """
    Compute P(replace | x, rc_bin) for every (x, rc_bin) state.

    No interpolation needed — v0 and v1 are already on the exact grid.
    Returns the full (N_X, N_RC_BINS) matrix of replacement probabilities.
    """
    # P(replace | x, j) = 1 / (1 + exp(v0 - v1))
    return 1.0 / (1.0 + np.exp(v0_grid - v1_grid))   # (7, 8)


# ── Compute at initial guess ──
P_pred_init = compute_choice_probs_grid(v0_init, v1_init)   # (7, 8)

print("Predicted P(replace | x, rc_bin)  at theta1=1, theta2=1:")
print(f"{'':6}", "  ".join(f"bin{j}" for j in range(N_RC_BINS)))
for xi in range(N_X):
    print(f"  x={xi+1}:  {np.round(P_pred_init[xi], 3)}")

print("\nEmpirical P(replace | x, rc_bin)  (from data frequency):")
for xi in range(N_X):
    print(f"  x={xi+1}:  {np.round(P_emp[xi], 3)}")

Predicted P(replace | x, rc_bin)  at theta1=1, theta2=1:
       bin0  bin1  bin2  bin3  bin4  bin5  bin6  bin7
  x=1:  [0. 0. 0. 0. 0. 0. 0. 0.]
  x=2:  [0. 0. 0. 0. 0. 0. 0. 0.]
  x=3:  [0. 0. 0. 0. 0. 0. 0. 0.]
  x=4:  [0. 0. 0. 0. 0. 0. 0. 0.]
  x=5:  [0. 0. 0. 0. 0. 0. 0. 0.]
  x=6:  [0. 0. 0. 0. 0. 0. 0. 0.]
  x=7:  [0. 0. 0. 0. 0. 0. 0. 0.]

Empirical P(replace | x, rc_bin)  (from data frequency):
  x=1:  [0.028 0.019 0.009 0.005 0.004 0.002 0.    0.   ]
  x=2:  [0.078 0.066 0.03  0.022 0.015 0.008 0.004 0.008]
  x=3:  [0.265 0.173 0.134 0.075 0.049 0.039 0.013 0.011]
  x=4:  [0.439 0.343 0.277 0.171 0.128 0.086 0.067 0.025]
  x=5:  [0.621 0.531 0.43  0.342 0.252 0.187 0.134 0.082]
  x=6:  [0.669 0.63  0.565 0.471 0.369 0.312 0.225 0.091]
  x=7:  [0.826 0.74  0.643 0.565 0.478 0.397 0.28  0.208]


### Log-Likelihood at the State Level

The likelihood collapses to a state-level sum:

$$\ell(\theta_1, \theta_2) = \sum_{s=1}^{S} \Bigl[
  \hat{P}(a{=}1 \mid x_s, rc_s,\, \theta)\cdot N_1(x_s, rc_s)
  + \hat{P}(a{=}0 \mid x_s, rc_s,\, \theta)\cdot N_0(x_s, rc_s)
\Bigr]$$

where $S = 56$ states, and $N_1, N_0$ are the count matrices.


In [12]:
def neg_log_likelihood(params):
    """
    Negative log-likelihood: choice component only.
    AR(1) params are pre-estimated and Pi is pre-built — neither appears here.

    params : [theta1, theta2]
    """
    theta1, theta2 = params

    # Guard rails: cost parameters must be positive
    if theta1 <= 0 or theta2 <= 0:
        return 1e10

    #  Step A: Bellman fixed point (Pi is fixed from data) 
    _, v0_grid, v1_grid = solve_bellman(theta1, theta2, rc_grid, Pi)

    #  Step B: Predicted P(replace) at every state 
    P_rep = compute_choice_probs_grid(v0_grid, v1_grid)   # (N_X, N_RC_BINS)
    P_mnt = 1.0 - P_rep                                   # P(maintain)

    #  Step C: State-level log-likelihood 
    # Sum over all 56 states: N1[x,j] * log P(rep) + N0[x,j] * log P(mnt)
    eps = 1e-12   # prevent log(0)
    ll = np.sum(N1 * np.log(P_rep + eps) + N0 * np.log(P_mnt + eps))

    return -ll   # return negative LL (scipy minimises)


#  Check at initial guess 
params_init = np.array([1.0, 1.0])
nll_init = neg_log_likelihood(params_init)
print(f"Negative log-likelihood at theta1=1, theta2=1:  {nll_init:.4f}")
print(f"(Log-likelihood = {-nll_init:.4f})")

Negative log-likelihood at theta1=1, theta2=1:  386763.3267
(Log-likelihood = -386763.3267)


### Optimization

The outer loop now searches over only $\theta_1$ and $\theta_2$.
$\Pi$, the AR(1) parameters, and the RC grid are all fixed.

In [13]:
print("--- Starting NFXP Optimisation (empirical Pi, 2 params) ---")
print(f"  Initial theta1 = {params_init[0]}")
print(f"  Initial theta2 = {params_init[1]}")
print(f"  Initial neg-LL = {neg_log_likelihood(params_init):.4f}\n")

t0_theta = time.perf_counter()
result = minimize(
    neg_log_likelihood,
    params_init,
    method="Nelder-Mead",
    options=dict(
        maxiter = 5000,
        xatol   = 1e-8,
        fatol   = 1e-8,
        disp    = True,
    )
)
theta_runtime_sec = time.perf_counter() - t0_theta
theta_runtime_ms = 1000.0 * theta_runtime_sec
running_time_sec = theta_runtime_sec
running_time_ms = theta_runtime_ms
print(f"\nComparable running time: {running_time_sec:.4f} seconds ({running_time_ms:.2f} ms)")

--- Starting NFXP Optimisation (empirical Pi, 2 params) ---
  Initial theta1 = 1.0
  Initial theta2 = 1.0
  Initial neg-LL = 386763.3267

Optimization terminated successfully.
         Current function value: 33977.621162
         Iterations: 76
         Function evaluations: 150

Comparable running time: 0.8212 seconds (821.22 ms)


### Results

In [14]:
theta1_hat, theta2_hat = result.x

print("=" * 55)
print("      NFXP ESTIMATION RESULTS")
print("=" * 55)
print(f"  theta1    (mileage cost coeff.)    = {theta1_hat:.4f}")
print(f"  theta2    (replacement cost coeff) = {theta2_hat:.4f}")
print(f"  ---  pre-estimated  ---")
print(f"  rho0      (AR(1) intercept)        = {rho0_ols:.4f}")
print(f"  rho1      (AR(1) persistence)      = {rho1_ols:.4f}")
print(f"  sigma_rho (AR(1) std dev)          = {sigma_rho_ols:.4f}")
print("-" * 55)
print(f"  Log-likelihood (choice only)       = {-result.fun:.4f}")
print(f"  Converged: {result.success}   |   Iterations: {result.nit}")
print(f"  Comparable running time            = {running_time_sec:.4f} sec ({running_time_ms:.2f} ms)")
print("=" * 55)

#print("\nComparison with professor Jingyuan:")
#print(f"  Professor:  theta1 = 0.4118   theta2 = 0.1567")
#print(f"  This code:  theta1 = {theta1_hat:.4f}   theta2 = {theta2_hat:.4f}")

      NFXP ESTIMATION RESULTS
  theta1    (mileage cost coeff.)    = 0.4088
  theta2    (replacement cost coeff) = 0.1563
  ---  pre-estimated  ---
  rho0      (AR(1) intercept)        = 17.8269
  rho1      (AR(1) persistence)      = 0.6249
  sigma_rho (AR(1) std dev)          = 4.6060
-------------------------------------------------------
  Log-likelihood (choice only)       = -33977.6212
  Converged: True   |   Iterations: 76
  Comparable running time            = 0.8212 sec (821.22 ms)


In [15]:
master_results['Nested Fixed Point'] = {
    'part': '(a)',
    'method': 'Nested Fixed Point',
    'theta1': float(theta1_hat),
    'theta2': float(theta2_hat),
    'running_time_sec': float(running_time_sec),
    'running_time_ms': float(running_time_ms),
}
master_common['Nested Fixed Point'] = {
    'rho0': float(rho0_ols),
    'rho1': float(rho1_ols),
    'sigma_rho': float(sigma_rho_ols),
}
print('Saved summary for Part (a) in master_results.')


Saved summary for Part (a) in master_results.


## Summary: What We Did (and Why)

### Algorithm flow

```
ONCE (before optimizer):
  1. Cut observed RC into 8 equal-width bins → rc_grid (bin midpoints)
  2. Count (bin_{t-1} → bin_t) transitions → count matrix C  →  Pi = C / row_sums
  3. OLS on (RC_{t-1}, RC_t) pairs → rho0, rho1, sigma_rho  (fixed)
  4. Count N1[x,j] and N0[x,j] for all 56 states

OUTER LOOP (Nelder-Mead over theta1, theta2 only):
  ├── INNER LOOP: Bellman iteration using fixed Pi and rc_grid
  │     EV_cont = EV @ Pi.T          ← matrix multiply
  │     v0 = u0 + β * EV_cont[x+1, :]
  │     v1 = u1 + β * EV_cont[1,   :]
  │     EV = log(exp(v0) + exp(v1)) + γ_E
  │     repeat until max|ΔEV| < 1e-8
  │
  ├── P(replace|x,j) = 1 / (1 + exp(v0 - v1))    ← no interpolation
  └── ℓ = Σ_{x,j} [N1[x,j] log P_rep + N0[x,j] log P_mnt]
```

### Why no Tauchen?

Tauchen is needed when your grid changes with parameters —
i.e., when the AR(1) parameters are inside the optimizer.
Here, $\Pi$ is built from the **data directly** and is never recomputed.
The grid is fixed, so no Tauchen formula is needed.

### Tradeoff
| | Empirical (this notebook) | Tauchen |
|---|---|---|
| Grid built from | Data (bins) | AR(1) theory |
| $\Pi$ recomputed? | Never | Every optimizer call |
| Params in optimizer | 2 | 5 |
| Empty-cell problem? | Yes (rare states) | No |
| Speed | Fast | Slower |

---
## Part (b): Hotz-Miller (1993) Inversion


# ECON 662 — Problem Set 2
## Part (b): Hotz-Miller (1993) Inversion — MLE

Estimate $(\theta, \rho, \sigma_\rho)$ without solving the value function in the estimation algorithm, but relying on the Hotz-Miller (1993) inversion. Estimate the parameters with MLE.

### The core idea 

In Part (a) NFXP, for each guess of $\theta$ the inner loop solves a fixed point:
$$V = \ln\!\Bigl(\sum_j \exp(\delta_j)\Bigr) + \gamma$$
where $V$ appears on both sides, requiring iterative contraction until convergence.

Under Type I EV errors, the conditional expectation of the shock given action $j$ is selected satisfies:
$$\phi_j(s) \equiv \mathbb{E}[\varepsilon_j \mid j \text{ selected}] = \gamma - \ln Pr_j^*(s)$$
This lets us write the ex-ante value function as a CCP-weighted average:
$$V(s) = \sum_{j \in J} Pr_j^*(s) \cdot \bigl(\delta_j(s) + \phi_j(s)\bigr)$$
where $\delta_j(s) = \bar{u}(s,j;\theta) + \beta\,EV(s,j)$ is the choice-specific value.

Expanding $EV(s,j) = \sum_{s'} TP(j,s)_{s,s'} V(s')$ and collecting $V$ on the left gives the Hotz-Miller linear system:

$$\boxed{V(\mathbf{s}) = \Bigl[I_{|S|\times|S|} - \beta \sum_{j} P_j(\mathbf{s}) \cdot TP(j,\mathbf{s})\Bigr]^{-1} \times \Bigl[\sum_{j} P_j(\mathbf{s}) \cdot \bigl(u_j(\mathbf{s}) + \phi_j(\mathbf{s})\bigr)\Bigr]}$$

**Notation:**
- $P_j(\mathbf{s})$: $|S|$-vector of $\hat{P}(a=j|s)$ : observed from data, fixed before optimizer
- $TP(j, \mathbf{s})$: $|S|\times|S|$ transition matrix conditional on action $j$ : built from data, fixed
- $u_j(\mathbf{s})$: $|S|$-vector of deterministic payoffs : depends on $\theta$
- $\phi_j(\mathbf{s}) = \gamma - \ln P_j(\mathbf{s})$: logit surplus : fixed from data
- $P_j(\mathbf{s}) \cdot TP(j,\mathbf{s})$: row-wise scaling (row $s$ of $TP$ scaled by scalar $P_j(s)$)

**Four-step procedure**
1. Estimate transition matrix : exogenous (RC, empirical counts) + endogenous (mileage)
2. Estimate CCP $\hat{P}(a=j|s)$ from data (frequency estimator)
3. For each guess of $\theta$: $V \leftarrow (\widehat{CCP},\,\theta)$ via HM inversion; get $P$ from $V$
4. Calculate likelihood given current $\theta$

**RC discretization: empirical bins**

Same as Part (a): we cut the observed RC range into $N_{RC}$ equal-width bins and represent each
by its midpoint. The transition matrix $\Pi$ is built by counting empirical transitions in the data.
This is fixed once and never recomputed during optimization, no parametric normal CDF, no re-evaluation at each optimizer call.

#### Import Libraries

In [16]:
import time
import numpy as np
import pandas as pd
from scipy.optimize import minimize

np.random.seed(42)

#### Load Data

In [17]:
# Variables: i (bus id), t (month 1-100), a (0=maintain, 1=replace),
#            x (mileage state 1-7), rc (replacement cost, continuous)
df = pd.read_csv("ddc_pset.csv")
df = df.sort_values(["i", "t"]).reset_index(drop=True)

print(f"Loaded {len(df):,} observations for {df['i'].nunique():,} buses.")
print(f"RC range: [{df['rc'].min():.4f}, {df['rc'].max():.4f}]")
print(f"Replacement rate: {df['a'].mean():.4f}")
df.head(10)

Loaded 100,000 observations for 1,000 buses.
RC range: [31.5000, 62.5000]
Replacement rate: 0.1724


,i,t,a,x,rc
0,1,1,0,4,45.0
1,1,2,0,5,49.0
2,1,3,0,6,54.0
3,1,4,1,7,47.0
4,1,5,0,1,56.0
5,1,6,0,2,52.5
6,1,7,0,3,55.0
7,1,8,0,4,50.5
8,1,9,0,5,52.0
9,1,10,0,6,42.0


#### Global Constants

In [18]:
BETA        = 0.95          # discount factor (given)
N_X         = 7             # mileage states {1,...,7}
N_RC_BINS   = 8             # RC bins (same as Part a)
S           = N_X * N_RC_BINS  # |S| = 56 total states
GAMMA       = 0.5772156649  # Euler-Mascheroni constant

# State ordering: s = xi * N_RC_BINS + j,  xi in {0,...,6},  j in {0,...,7}
print(f"|S| = {N_X} x {N_RC_BINS} = {S} states")

|S| = 7 x 8 = 56 states


### Estimate Transition Matrix 

Same approach as Part (a). We:

1. Cut the observed RC range into $N_{RC}=8$ equal-width bins; each bin represented by its midpoint.
2. Count empirical transitions $(bin_{t-1} \to bin_t)$ within each bus to get count matrix $C$.
3. Normalize rows: $\Pi_{j,j'} = C_{j,j'} / \sum_{j''} C_{j,j''}$.

$\Pi$ is built once from the data and never touched again during optimization.

We also estimate $(\rho_0, \rho_1, \sigma_\rho)$ via OLS — the AR(1) likelihood is separable
from the choice likelihood, so these are fixed before the optimizer runs.

In [19]:
# Equal-width bins over [RC_min, RC_max]  (same as Part a)
rc_min      = df['rc'].min()
rc_max      = df['rc'].max()
bin_edges   = np.linspace(rc_min, rc_max, N_RC_BINS + 1)        # shape (9,)
bin_midpoints = (bin_edges[:-1] + bin_edges[1:]) / 2.0           # shape (8,)
rc_grid     = bin_midpoints   # alias: representative RC value per bin

# Assign each observation to its bin (0-indexed)
rc_bin_idx  = np.searchsorted(bin_edges[1:], df['rc'].values, side='left')
rc_bin_idx  = np.clip(rc_bin_idx, 0, N_RC_BINS - 1)
df['rc_bin'] = rc_bin_idx
df['rc_mid'] = bin_midpoints[rc_bin_idx]

print(f"Bin edges: {np.round(bin_edges, 3)}")
print(f"Bin midpoints (rc_grid): {np.round(rc_grid, 3)}")

Bin edges: [31.5   35.375 39.25  43.125 47.    50.875 54.75  58.625 62.5  ]
Bin midpoints (rc_grid): [33.438 37.312 41.188 45.062 48.938 52.812 56.688 60.562]


In [20]:
# Build empirical transition count matrix C (N_RC_BINS x N_RC_BINS)
# C[j, j'] = # times RC moved from bin j to bin j' in consecutive periods
C = np.zeros((N_RC_BINS, N_RC_BINS))
for _, g in df.groupby("i"):
    bins = g['rc_bin'].values
    for j_from, j_to in zip(bins[:-1], bins[1:]):
        C[j_from, j_to] += 1

# Normalize rows -> empirical transition probability matrix Pi
Pi = C / C.sum(axis=1, keepdims=True)   # shape (8, 8)

print(f"Pi ({N_RC_BINS}x{N_RC_BINS}) — built from data, FIXED for entire estimation:")
print(np.round(Pi, 4))
print(f"Row sums: {np.round(Pi.sum(axis=1), 10)}")

Pi (8x8) — built from data, FIXED for entire estimation:
[[0.     0.3333 0.6667 0.     0.     0.     0.     0.    ]
 [0.2    0.4    0.2    0.2    0.     0.     0.     0.    ]
 [0.1333 0.     0.4    0.2    0.2    0.0667 0.     0.    ]
 [0.     0.087  0.087  0.3043 0.3913 0.087  0.0435 0.    ]
 [0.     0.     0.1154 0.2692 0.2692 0.3077 0.0385 0.    ]
 [0.     0.     0.0556 0.2778 0.2778 0.1111 0.2222 0.0556]
 [0.     0.     0.     0.     0.2857 0.5714 0.     0.1429]
 [0.     0.     0.     0.     0.     0.5    0.5    0.    ]]
Row sums: [1. 1. 1. 1. 1. 1. 1. 1.]


In [21]:
# OLS for AR(1) parameters (separable from choice likelihood, estimated once)
# RC_t = rho0 + rho1 * RC_{t-1} + e_t,   beta_hat = (X'X)^{-1} X'y
rc_prev_list, rc_curr_list = [], []
for _, g in df.groupby("i"):
    v = g['rc'].values
    rc_prev_list.append(v[:-1])
    rc_curr_list.append(v[1:])
rc_prev = np.concatenate(rc_prev_list)
rc_curr = np.concatenate(rc_curr_list)

X_ols    = np.column_stack([np.ones(len(rc_prev)), rc_prev])
beta_ols = np.linalg.solve(X_ols.T @ X_ols, X_ols.T @ rc_curr)
rho0, rho1 = beta_ols
sigma_rho  = (rc_curr - X_ols @ beta_ols).std(ddof=1)

print(f"rho0={rho0:.4f}, rho1={rho1:.4f}, sigma_rho={sigma_rho:.4f}")

rho0=17.8269, rho1=0.6249, sigma_rho=4.6060


#### Build $TP(j, \mathbf{s})$: Full $|S|\times|S|$ Action-Specific Transition Matrices

The combined state is $s = (x_i, RC_j)$, indexed as $s = x_i \times N_{RC} + j$ ($S=56$ states).

From the mileage law of motion:
- $a=0$ (maintain): $x \to \min(x+1, 7)$, RC transitions via $\Pi$
- $a=1$ (replace): $x \to 1$, RC transitions via $\Pi$

So the action-specific full transition is:
$$TP(a, s)_{s,\,s'} = \mathbf{1}[x' = x^a_{\text{next}}] \times \Pi_{j,j'}$$

Row $s$ of $TP_a$ places the $\Pi$ row corresponding to the current RC bin $j$
into the block of columns for next mileage $x^a_{\text{next}}$.

In [22]:
# State-level arrays (length S=56): mileage and RC at each flat state index
xi_of_s = np.array([xi for xi in range(N_X) for _ in range(N_RC_BINS)])  # mileage index 0..6
j_of_s  = np.array([j  for _  in range(N_X) for j in range(N_RC_BINS)])  # RC bin index 0..7
x_of_s  = xi_of_s + 1          # actual mileage x = 1..7
rc_of_s = rc_grid[j_of_s]      # RC midpoint at each state

# Next mileage index after each action (vectorized over all S states)
xi_next_mnt = np.minimum(xi_of_s + 1, N_X - 1)   # maintain: min(xi+1, 6)
xi_next_rep = np.zeros(S, dtype=int)               # replace: always xi=0 (x=1)

# Build TP0 (S x S): transition matrix under action a=0 (maintain)
# Row s: place Pi[j_of_s[s], :] at columns xi_next_mnt[s]*N_RC_BINS .. +N_RC_BINS
TP0 = np.zeros((S, S))
for s in range(S):
    col_start = xi_next_mnt[s] * N_RC_BINS
    TP0[s, col_start : col_start + N_RC_BINS] = Pi[j_of_s[s], :]

# Build TP1 (S x S): transition matrix under action a=1 (replace)
# Row s: place Pi[j_of_s[s], :] at columns 0..N_RC_BINS-1 (next mileage always xi=0)
TP1 = np.zeros((S, S))
for s in range(S):
    TP1[s, 0 : N_RC_BINS] = Pi[j_of_s[s], :]

print(f"TP0 shape: {TP0.shape}  (maintain, a=0)")
print(f"TP1 shape: {TP1.shape}  (replace,  a=1)")
print(f"TP0 row sums: [{TP0.sum(1).min():.8f}, {TP0.sum(1).max():.8f}]")
print(f"TP1 row sums: [{TP1.sum(1).min():.8f}, {TP1.sum(1).max():.8f}]")

TP0 shape: (56, 56)  (maintain, a=0)
TP1 shape: (56, 56)  (replace,  a=1)
TP0 row sums: [1.00000000, 1.00000000]
TP1 row sums: [1.00000000, 1.00000000]


### Estimate $\hat{P}(a=j|s)$ : Conditional Choice Probabilities (CCP)

**Frequency estimator** (from professor Jingyuan's notes):
$$\hat{P}(a=1|s) = \frac{N_1(s)}{N_1(s) + N_0(s)}, \qquad \hat{P}(a=0|s) = 1 - \hat{P}(a=1|s)$$

These are estimated once and held completely fixed during MLE. This is what eliminates the inner fixed-point loop,  we observe $P$ from data rather than computing it from $V$.

We also build the logit surplus $\phi_j(s) = \gamma - \ln \hat{P}_j(s)$ .
This captures $\mathbb{E}[\varepsilon_j \mid j\text{ chosen}]$ under Type I EV and is the key term that allows us to write $V$ without iterating.

In [23]:
# Count N1 and N0 at each of the S=56 states
# x is 1-indexed in data; xi_obs = x-1 is 0-indexed
N1 = np.zeros((N_X, N_RC_BINS))
N0 = np.zeros((N_X, N_RC_BINS))
for xi_obs, j_obs, a_obs in zip(df['x'].values.astype(int) - 1,
                                 df['rc_bin'].values.astype(int),
                                 df['a'].values.astype(int)):
    if a_obs == 1:
        N1[xi_obs, j_obs] += 1
    else:
        N0[xi_obs, j_obs] += 1

N_total = N1 + N0

# Frequency CCP — clip to (eps, 1-eps) so log(P) is always finite
eps    = 1e-6
P1_hat = np.where(N_total > 0, N1 / N_total, eps)
P1_hat = np.clip(P1_hat, eps, 1 - eps)   # shape (7, 8)
P0_hat = 1.0 - P1_hat

print("Frequency CCP  P_hat(a=1 | x, rc_bin):")
print("       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7")
for xi in range(N_X):
    print(f"  x={xi+1}: {np.round(P1_hat[xi], 3)}")
print(f"\nTotal obs: {N_total.sum():.0f}")

Frequency CCP  P_hat(a=1 | x, rc_bin):
       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7
  x=1: [0.028 0.019 0.009 0.005 0.004 0.002 0.    0.   ]
  x=2: [0.078 0.066 0.03  0.022 0.015 0.008 0.004 0.008]
  x=3: [0.265 0.173 0.134 0.075 0.049 0.039 0.013 0.011]
  x=4: [0.439 0.343 0.277 0.171 0.128 0.086 0.067 0.025]
  x=5: [0.621 0.531 0.43  0.342 0.252 0.187 0.134 0.082]
  x=6: [0.669 0.63  0.565 0.471 0.369 0.312 0.225 0.091]
  x=7: [0.826 0.74  0.643 0.565 0.478 0.397 0.28  0.208]

Total obs: 100000


### Pre-compute Fixed Objects (Outside the Optimizer)

From the HM formula , decompose what depends on $\theta$ vs. what is fixed:

$$V = \underbrace{\Bigl[I - \beta\sum_j P_j \cdot TP_j\Bigr]^{-1}}_{A^{-1},\ \text{fixed}} \times \underbrace{\Bigl[\sum_j P_j \odot (u_j + \phi_j)\Bigr]}_{\mathbf{b}(\theta),\ \text{changes with }\theta}$$

| Object | Fixed? | Expression |
|---|---|---|
| $\phi_j(s) = \gamma - \ln \hat{P}_j(s)$ | Yes | logit surplus |
| $P_j(s) \cdot TP_j$ | Yes | row-weighted TP matrices |
| $A = I - \beta(P_0 \cdot TP_0 + P_1 \cdot TP_1)$ | **Yes — pre-compute once** | $(S\times S)$ matrix |
| $u_j(s;\theta)$ | No | $-\theta_1 x$, $-\theta_2 RC$ |
| $\mathbf{b}(\theta)$ | No | $P_0 \odot(u_0+\phi_0) + P_1 \odot(u_1+\phi_1)$ |

Row-scaling notation: $P_j(s) \cdot TP_j$ means each row $s$ of $TP_j$ is multiplied by the scalar $\hat{P}(a=j|s)$. In numpy: `P_j_flat[:, None] * TP_j`.

In [24]:
# Flatten CCP to length-S vectors (ordering: s = xi*N_RC_BINS + j)
P1_flat = P1_hat.ravel()   # (56,)  P(replace | state s)
P0_flat = P0_hat.ravel()   # (56,)  P(maintain | state s)

# Logit surplus phi_j(s) = gamma - ln P_j(s)  [slide 21]
# = E[epsilon_j | action j is chosen]  under Type I EV
phi1 = GAMMA - np.log(P1_flat)   # (56,): surplus under replace
phi0 = GAMMA - np.log(P0_flat)   # (56,): surplus under maintain

# Row-scaled transition matrices: P_j(s) * TP_j
# P_j_flat[:, None] broadcasts to (S,1); multiplying by TP_j (S,S)
# scales each row s of TP_j by the scalar P_j(s)
P0_TP0 = P0_flat[:, None] * TP0   # (56, 56)
P1_TP1 = P1_flat[:, None] * TP1   # (56, 56)

# A = I - beta * [P0*TP0 + P1*TP1]   (slide 22)
# This matrix is FIXED for the entire optimization — built once here
A = np.eye(S) - BETA * (P0_TP0 + P1_TP1)   # (56, 56)

# Sanity: the mixed transition (P0*TP0 + P1*TP1) rows must sum to 1
mixed_rowsums = (P0_TP0 + P1_TP1).sum(axis=1)
print(f"Mixed transition row sums: [{mixed_rowsums.min():.8f}, {mixed_rowsums.max():.8f}]")
print(f"Matrix A shape: {A.shape}  : pre-computed, FIXED during entire MLE")

Mixed transition row sums: [1.00000000, 1.00000000]
Matrix A shape: (56, 56)  : pre-computed, FIXED during entire MLE


### Inside Optimizer: $V \leftarrow (\widehat{CCP},\,\theta)$ via HM Inversion

*Guess $\theta$, get $V$ via HM inversion; then get $P$ from $V$*

For each optimizer guess of $\theta = (\theta_1, \theta_2)$:

**Deterministic payoffs** (only $\theta$-dependent objects):
$$u_0(s;\theta) = -\theta_1 x_s \qquad u_1(s;\theta) = -\theta_2\, RC_s$$

**RHS vector** $\mathbf{b}(\theta)$ (element-wise over states):
$$b(s;\theta) = \hat{P}_0(s)\bigl(u_0(s;\theta)+\phi_0(s)\bigr) + \hat{P}_1(s)\bigl(u_1(s;\theta)+\phi_1(s)\bigr)$$

**HM inversion**: solve $A\,V = \mathbf{b}(\theta)$ : one linear system, no iteration.

**Recover model CCPs** (get $P$ from approximated $V$):
$$EV(s,j) = \bigl(TP_j \cdot V\bigr)_s \qquad \delta_j(s) = u_j(s;\theta) + \beta\,EV(s,j)$$
$$\tilde{P}(a=1|s;\theta) = \frac{e^{\delta_1(s)}}{e^{\delta_0(s)} + e^{\delta_1(s)}}$$
This matches solution notes eq. (2): $\hat{P}(a=1|x_s,rc_s;\theta) = \frac{\exp(-\theta_2 rc_s + EV(s,1))}{\exp(-\theta_1 x_s + EV(s,0)) + \exp(-\theta_2 rc_s + EV(s,1))}$

In [25]:
def hm_inversion(theta1, theta2):
    """
    HM inversion: V = A^{-1} b(theta)   [lecture slide 22]

    A = I - beta*(P0*TP0 + P1*TP1)  : pre-computed, FIXED
    b = P0*(u0+phi0) + P1*(u1+phi1) : built here, changes with theta

    u0(s) = -theta1 * x_of_s   (mileage cost)
    u1(s) = -theta2 * rc_of_s  (replacement cost)

    Returns V : ndarray (S,) = (56,)
    """
    # 3a: deterministic payoffs — only objects that depend on theta
    u0 = -theta1 * x_of_s    # (56,)
    u1 = -theta2 * rc_of_s   # (56,)

    # 3b: RHS vector b(theta) = sum_j P_j * (u_j + phi_j), element-wise
    b = P0_flat * (u0 + phi0) + P1_flat * (u1 + phi1)   # (56,)

    # 3c: solve A * V = b  — one exact linear solve (LU factorization)
    # A is pre-computed and never changes; b changes each call
    V = np.linalg.solve(A, b)   # (56,)

    return V


def model_ccp(theta1, theta2, V):
    """
    Recover model-implied P_tilde(a=1|s; theta) from HM-inverted V.

    EV(s,j) = TP_j @ V             : matrix-vector products (S^2)
    delta_j  = u_j(theta) + beta*EV_j
    P_tilde  = softmax(delta1, delta0)  : logit formula

    Returns P1_mod : ndarray (N_X, N_RC_BINS) = (7, 8)
    """
    u0 = -theta1 * x_of_s
    u1 = -theta2 * rc_of_s

    # 3d: action-specific continuation values  EV(s,j) = TP_j @ V
    EV0 = TP0 @ V   # (56,): expected continuation under maintain
    EV1 = TP1 @ V   # (56,): expected continuation under replace

    # Choice-specific values: delta_j(s) = u_j + beta * EV(s,j)
    delta0 = u0 + BETA * EV0   # (56,)
    delta1 = u1 + BETA * EV1   # (56,)

    # Logit CCP — numerically stable (subtract max before exp)
    d_max  = np.maximum(delta0, delta1)
    exp0   = np.exp(delta0 - d_max)
    exp1   = np.exp(delta1 - d_max)
    P1_mod = exp1 / (exp0 + exp1)   # (56,)

    return P1_mod.reshape(N_X, N_RC_BINS)   # (7, 8)


# Inspect at initial guess theta1=theta2=1
V_test   = hm_inversion(1.0, 1.0)
P1_test  = model_ccp(1.0, 1.0, V_test)

print("V from HM inversion at theta1=theta2=1 (7x8 grid):")
print(np.round(V_test.reshape(N_X, N_RC_BINS), 2))
print("\nModel CCP P_tilde(replace|x,j) at theta1=theta2=1:")
for xi in range(N_X):
    print(f"  x={xi+1}: {np.round(P1_test[xi], 3)}")
print("\nData frequency CCP P_hat(replace|x,j):")
for xi in range(N_X):
    print(f"  x={xi+1}: {np.round(P1_hat[xi], 3)}")

V from HM inversion at theta1=theta2=1 (7x8 grid):
[[-176.67 -176.6  -176.07 -175.79 -175.62 -175.52 -175.28 -175.15]
 [-185.24 -185.22 -184.79 -184.44 -184.34 -183.99 -183.87 -183.8 ]
 [-192.51 -192.07 -192.68 -192.27 -192.06 -191.99 -191.31 -191.34]
 [-196.5  -197.03 -198.42 -198.61 -198.77 -198.49 -198.53 -197.7 ]
 [-198.59 -200.09 -202.04 -203.44 -203.87 -203.85 -203.87 -203.63]
 [-199.38 -201.55 -204.15 -206.12 -207.04 -207.74 -207.94 -206.25]
 [-199.86 -202.51 -205.08 -207.39 -208.83 -209.61 -209.63 -209.42]]

Model CCP P_tilde(replace|x,j) at theta1=theta2=1:
  x=1: [0. 0. 0. 0. 0. 0. 0. 0.]
  x=2: [0. 0. 0. 0. 0. 0. 0. 0.]
  x=3: [0. 0. 0. 0. 0. 0. 0. 0.]
  x=4: [0.004 0.    0.    0.    0.    0.    0.    0.   ]
  x=5: [0.06  0.001 0.    0.    0.    0.    0.    0.   ]
  x=6: [0.297 0.004 0.001 0.    0.    0.    0.    0.   ]
  x=7: [0.534 0.01  0.003 0.    0.    0.    0.    0.   ]

Data frequency CCP P_hat(replace|x,j):
  x=1: [0.028 0.019 0.009 0.005 0.004 0.002 0.    0.   ]
  x

### MLE : Calculate Likelihood Given $\theta$

*Calculate likelihood given current $\theta$*

**Objective function** :
$$\ell(\theta_1, \theta_2) = \sum_{s=1}^{|S|} \Bigl[ N_1(s)\, \ln\tilde{P}(a{=}1|s;\theta) + N_0(s)\, \ln\tilde{P}(a{=}0|s;\theta) \Bigr]$$

**Same objective as Part (a).** Only how $\tilde{P}$ is computed differs:
- Part (a): Bellman contraction inside optimizer (50–200 iterations per call)
- Part (b): $A\,V = \mathbf{b}(\theta)$ — one linear solve per call


In [26]:
def neg_log_likelihood(params):
    """
    Negative log-likelihood for Hotz-Miller MLE (slide 23, steps 3-4).

    Per optimizer call:
      1. hm_inversion(theta)       ->  V      [one linear solve: A V = b(theta)]
      2. model_ccp(theta, V)       ->  P_tilde [EV = TP_j @ V, then logit]
      3. LL = sum_s[N1 log P1 + N0 log P0]

    Fixed objects (never recomputed): A, TP0, TP1, phi0, phi1,
                                      P0_flat, P1_flat, N0, N1
    """
    theta1, theta2 = params
    if theta1 <= 0 or theta2 <= 0:
        return 1e10

    V     = hm_inversion(theta1, theta2)      # (56,)
    P1mod = model_ccp(theta1, theta2, V)      # (7, 8)
    P0mod = 1.0 - P1mod

    eps = 1e-12
    ll  = np.sum(N1 * np.log(P1mod + eps) + N0 * np.log(P0mod + eps))
    return -ll


params_init = np.array([1.0, 1.0])
print(f"NLL at theta1=theta2=1: {neg_log_likelihood(params_init):.4f}")

NLL at theta1=theta2=1: 213473.6930


In [27]:
# MLE outer loop — Nelder-Mead, same as Part (a) for fair comparison
print("--- Hotz-Miller MLE (steps 3-4 in optimizer loop) ---")
print(f"  theta_init={params_init},  NLL_init={neg_log_likelihood(params_init):.4f}\n")

t0_theta = time.perf_counter()
result = minimize(
    neg_log_likelihood,
    params_init,
    method="Nelder-Mead",
    options=dict(maxiter=5000, xatol=1e-8, fatol=1e-8, disp=True)
)
theta_runtime_sec = time.perf_counter() - t0_theta
theta_runtime_ms = 1000.0 * theta_runtime_sec
running_time_sec = theta_runtime_sec
running_time_ms = theta_runtime_ms
print(f"\nComparable running time: {running_time_sec:.4f} seconds ({running_time_ms:.2f} ms)")

--- Hotz-Miller MLE (steps 3-4 in optimizer loop) ---
  theta_init=[1. 1.],  NLL_init=213473.6930

Optimization terminated successfully.
         Current function value: 33977.394964
         Iterations: 78
         Function evaluations: 146

Comparable running time: 0.0081 seconds (8.06 ms)


---
### Results

In [28]:
theta1_hat, theta2_hat = result.x

print("=" * 58)
print("   HOTZ-MILLER (1993) INVERSION -- MLE RESULTS")
print("=" * 58)
print(f"  theta1    (mileage cost coeff.)     = {theta1_hat:.4f}")
print(f"  theta2    (replacement cost coeff)  = {theta2_hat:.4f}")
print("  ---  pre-estimated (OLS, separable)  ---")
print(f"  rho0      (AR(1) intercept)         = {rho0:.4f}")
print(f"  rho1      (AR(1) persistence)       = {rho1:.4f}")
print(f"  sigma_rho (AR(1) std dev)           = {sigma_rho:.4f}")
print("-" * 58)
print(f"  Log-likelihood (choice component)   = {-result.fun:.4f}")
print(f"  Converged: {result.success}   Iterations: {result.nit}")
print(f"  Comparable running time             = {running_time_sec:.4f} sec ({running_time_ms:.2f} ms)")
print("=" * 58)
print()
#print("Benchmark (solution notes from Professor Jingyuan):")
#print("  theta1=0.4118, theta2=0.1567, time=7.32ms")

   HOTZ-MILLER (1993) INVERSION -- MLE RESULTS
  theta1    (mileage cost coeff.)     = 0.4088
  theta2    (replacement cost coeff)  = 0.1563
  ---  pre-estimated (OLS, separable)  ---
  rho0      (AR(1) intercept)         = 17.8269
  rho1      (AR(1) persistence)       = 0.6249
  sigma_rho (AR(1) std dev)           = 4.6060
----------------------------------------------------------
  Log-likelihood (choice component)   = -33977.3950
  Converged: True   Iterations: 78
  Comparable running time             = 0.0081 sec (8.06 ms)



In [29]:
master_results['Hotz and Miller'] = {
    'part': '(b)',
    'method': 'Hotz and Miller',
    'theta1': float(theta1_hat),
    'theta2': float(theta2_hat),
    'running_time_sec': float(running_time_sec),
    'running_time_ms': float(running_time_ms),
}
master_common['Hotz and Miller'] = {
    'rho0': float(rho0),
    'rho1': float(rho1),
    'sigma_rho': float(sigma_rho),
}
print('Saved summary for Part (b) in master_results.')


Saved summary for Part (b) in master_results.


### Part (a) NFXP vs Part (b) HM

| Dimension | Part (a) NFXP | Part (b) HM |
|---|---|---|
| **RC discretization** | Equal-width empirical bins | **Same** |
| **Pi** | Empirical counts, fixed | **Same** |
| **V computation** | Bellman contraction (50-200 iter) | One solve $AV=b(\theta)$ |
| **Inner loop?** | Yes | None |
| **CCP pre-estimated?** | No (computed from $V$) | Yes (fixed from data) |
| **$A$ pre-computed?** | No | Yes (outside loop) |
| **Speed** | 930 ms | ~7 ms (~100x faster) |
| **CCP noise** | None (self-consistent) | First-step error from $\hat{P}$ |
| **Logit assumption** | Both require Type I EV | HM uses $\phi_j = \gamma - \ln P_j$ critically |

---
## Part (c.1): Forward Simulation Using Equation (5)


# ECON 662 — Problem Set 2
## Part (c.1): Forward Simulation MLE Using Equation (5)

Estimate $(\theta,\rho,\sigma_\rho)$ using forward simulation to approximate the value function, instead of:
- solving a Bellman fixed point as in Part (a), or
- using the Hotz-Miller matrix inversion as in Part (b).

Following the homework notes and Lecture 15, the first two steps are the same as Part (b):

1. Estimate the replacement-cost process and the state transition objects.
2. Estimate the conditional choice probabilities (CCPs) from the data.

Then, for each state $s=(x,RC)$, approximate the value function by forward simulation using Equation (5):

$$
\hat V(s;\theta)
=
\frac{1}{N_{\text{sim}}}
\sum_{n=1}^{N_{\text{sim}}}
\sum_{\tau=0}^{T_{\text{sim}}-1}
\beta^\tau
\Bigl[
\bar u(s_{n\tau},a_{n\tau};\theta)
\;+\;
\gamma
\;-\;
\ln \hat P(a_{n\tau}\mid s_{n\tau})
\Bigr],
$$

where
- $a_{n\tau}$ is drawn from the estimated CCP $\hat P(\cdot\mid s_{n\tau})$,
- $s_{n,\tau+1}$ is drawn from the estimated transition process,
- $\bar u$ is the deterministic payoff component,
- $\gamma \approx 0.5772$ is Euler's constant.

#### Import Libraries

In [30]:
import time
import numpy as np
import pandas as pd
from scipy.optimize import minimize

np.random.seed(42)

#### Load Data


In [31]:
# Variables:
# i  = bus id
# t  = month
# a  = action (0 = maintain, 1 = replace)
# x  = mileage state in {1, ..., 7}
# rc = replacement cost (continuous)
df = pd.read_csv("ddc_pset.csv")
df = df.sort_values(["i", "t"]).reset_index(drop=True)

print(f"Loaded {len(df):,} observations for {df['i'].nunique():,} buses.")
print(f"RC range: [{df['rc'].min():.4f}, {df['rc'].max():.4f}]")
print(f"Replacement rate: {df['a'].mean():.4f}")
df.head(10)


Loaded 100,000 observations for 1,000 buses.
RC range: [31.5000, 62.5000]
Replacement rate: 0.1724


,i,t,a,x,rc
0,1,1,0,4,45.0
1,1,2,0,5,49.0
2,1,3,0,6,54.0
3,1,4,1,7,47.0
4,1,5,0,1,56.0
5,1,6,0,2,52.5
6,1,7,0,3,55.0
7,1,8,0,4,50.5
8,1,9,0,5,52.0
9,1,10,0,6,42.0


#### Global Constants


In [32]:
BETA        = 0.95
N_X         = 7
N_RC_BINS   = 8
S           = N_X * N_RC_BINS
GAMMA       = 0.5772156649

# Forward-simulation tuning parameters for Part (c.1)
N_SIM       = 100
T_SIM       = 120
SIM_SEED    = 12345

print(f"beta        = {BETA}")
print(f"mileage x   = {N_X} states")
print(f"RC bins     = {N_RC_BINS}")
print(f"|S|         = {S}")
print(f"N_SIM       = {N_SIM}")
print(f"T_SIM       = {T_SIM}")
print(f"beta^T_SIM  = {BETA**T_SIM:.6f}")


beta        = 0.95
mileage x   = 7 states
RC bins     = 8
|S|         = 56
N_SIM       = 100
T_SIM       = 120
beta^T_SIM  = 0.002122


### Estimate the Transition Objects and $(\rho_0,\rho_1,\sigma_\rho)$

This step is the same as Part (b).

We first discretize observed replacement costs into equal-width bins:

$$
RC \in [RC_{\min}, RC_{\max}] \longrightarrow \text{8 empirical bins}.
$$

Let $\Pi_{j,j'}$ be the empirical probability that $RC$ moves from bin $j$ today to bin $j'$ tomorrow.
This gives us the exogenous transition for the replacement-cost state.

Separately, we estimate the continuous AR(1) process

$$
RC_t = \rho_0 + \rho_1 RC_{t-1} + e_t, \qquad e_t \sim N(0,\sigma_\rho^2),
$$

by OLS using the observed continuous $RC$ values.

As in Parts (a) and (b), the Gaussian likelihood for the AR(1) process is separable from the choice likelihood, so we estimate $(\rho_0,\rho_1,\sigma_\rho)$ once before the MLE over $\theta$.


In [33]:
# Equal-width bins over the observed RC support
rc_min = df['rc'].min()
rc_max = df['rc'].max()
bin_edges = np.linspace(rc_min, rc_max, N_RC_BINS + 1)
rc_grid = (bin_edges[:-1] + bin_edges[1:]) / 2.0

# Assign each observation to a bin (0-indexed)
rc_bin_idx = np.searchsorted(bin_edges[1:], df['rc'].values, side='left')
rc_bin_idx = np.clip(rc_bin_idx, 0, N_RC_BINS - 1)
df['rc_bin'] = rc_bin_idx
df['rc_mid'] = rc_grid[rc_bin_idx]

print("RC bin edges:")
print(np.round(bin_edges, 3))
print("\nRC grid (bin midpoints):")
print(np.round(rc_grid, 3))

# Empirical RC transition matrix Pi
C = np.zeros((N_RC_BINS, N_RC_BINS))
for _, g in df.groupby("i"):
    bins = g['rc_bin'].values
    for j_from, j_to in zip(bins[:-1], bins[1:]):
        C[j_from, j_to] += 1

Pi = C / C.sum(axis=1, keepdims=True)

print("\nEmpirical RC transition matrix Pi:")
print(np.round(Pi, 4))
print("Row sums:")
print(np.round(Pi.sum(axis=1), 10))

# OLS for the continuous AR(1): RC_t = rho0 + rho1 * RC_{t-1} + e_t
rc_prev_list, rc_curr_list = [], []
for _, g in df.groupby("i"):
    vals = g['rc'].values
    rc_prev_list.append(vals[:-1])
    rc_curr_list.append(vals[1:])

rc_prev = np.concatenate(rc_prev_list)
rc_curr = np.concatenate(rc_curr_list)

X_ols = np.column_stack([np.ones(len(rc_prev)), rc_prev])
beta_ols = np.linalg.solve(X_ols.T @ X_ols, X_ols.T @ rc_curr)
rho0_hat, rho1_hat = beta_ols
sigma_rho_hat = (rc_curr - X_ols @ beta_ols).std(ddof=1)

print("\nAR(1) estimates from continuous RC:")
print(f"rho0_hat      = {rho0_hat:.4f}")
print(f"rho1_hat      = {rho1_hat:.4f}")
print(f"sigma_rho_hat = {sigma_rho_hat:.4f}")


RC bin edges:
[31.5   35.375 39.25  43.125 47.    50.875 54.75  58.625 62.5  ]

RC grid (bin midpoints):
[33.438 37.312 41.188 45.062 48.938 52.812 56.688 60.562]

Empirical RC transition matrix Pi:
[[0.     0.3333 0.6667 0.     0.     0.     0.     0.    ]
 [0.2    0.4    0.2    0.2    0.     0.     0.     0.    ]
 [0.1333 0.     0.4    0.2    0.2    0.0667 0.     0.    ]
 [0.     0.087  0.087  0.3043 0.3913 0.087  0.0435 0.    ]
 [0.     0.     0.1154 0.2692 0.2692 0.3077 0.0385 0.    ]
 [0.     0.     0.0556 0.2778 0.2778 0.1111 0.2222 0.0556]
 [0.     0.     0.     0.     0.2857 0.5714 0.     0.1429]
 [0.     0.     0.     0.     0.     0.5    0.5    0.    ]]
Row sums:
[1. 1. 1. 1. 1. 1. 1. 1.]

AR(1) estimates from continuous RC:
rho0_hat      = 17.8269
rho1_hat      = 0.6249
sigma_rho_hat = 4.6060


### Build Full State Transitions and Estimate CCPs

The combined state is

$$
s = (x, j),
$$

where $x \in \{1,\dots,7\}$ is mileage and $j \in \{0,\dots,7\}$ is the RC bin.
Hence there are

$$
|S| = 7 \times 8 = 56
$$

discrete states.

The action-specific state transitions are the same as in Part (b):

- If $a=0$ (maintain), then $x' = \min(x+1,7)$.
- If $a=1$ (replace), then $x' = 1$.
- In both cases, the RC-bin transition is governed by the empirical matrix $\Pi$.

We also estimate the CCP from observed state frequencies:

$$
\hat P(a=1\mid s) = \frac{N_1(s)}{N_0(s) + N_1(s)},
\qquad
\hat P(a=0\mid s) = 1 - \hat P(a=1\mid s).
$$

To avoid $\log(0)$ in Equation (5), we clip these probabilities slightly away from 0 and 1.


In [34]:
# Flat state indexing: s = xi * N_RC_BINS + j
xi_of_s = np.array([xi for xi in range(N_X) for _ in range(N_RC_BINS)])
j_of_s  = np.array([j  for _ in range(N_X) for j in range(N_RC_BINS)])

x_of_s  = xi_of_s + 1
rc_of_s = rc_grid[j_of_s]

# Next mileage index after each action
xi_next_mnt = np.minimum(xi_of_s + 1, N_X - 1)
xi_next_rep = np.zeros(S, dtype=int)

# Full action-specific transition matrices TP0 and TP1, shape (56, 56)
TP0 = np.zeros((S, S))
TP1 = np.zeros((S, S))

for s in range(S):
    j = j_of_s[s]

    col0 = xi_next_mnt[s] * N_RC_BINS
    TP0[s, col0:col0 + N_RC_BINS] = Pi[j, :]

    col1 = xi_next_rep[s] * N_RC_BINS
    TP1[s, col1:col1 + N_RC_BINS] = Pi[j, :]

print(f"TP0 shape: {TP0.shape}")
print(f"TP1 shape: {TP1.shape}")
print(f"TP0 row sums: [{TP0.sum(1).min():.8f}, {TP0.sum(1).max():.8f}]")
print(f"TP1 row sums: [{TP1.sum(1).min():.8f}, {TP1.sum(1).max():.8f}]")

# State-level counts
N1 = np.zeros((N_X, N_RC_BINS))
N0 = np.zeros((N_X, N_RC_BINS))

for xi_obs, j_obs, a_obs in zip(df['x'].values.astype(int) - 1,
                                df['rc_bin'].values.astype(int),
                                df['a'].values.astype(int)):
    if a_obs == 1:
        N1[xi_obs, j_obs] += 1
    else:
        N0[xi_obs, j_obs] += 1

N_total = N0 + N1

# Smoothed/clipped CCP
eps_ccp = 1e-6
P1_hat = np.where(N_total > 0, N1 / N_total, eps_ccp)
P1_hat = np.clip(P1_hat, eps_ccp, 1.0 - eps_ccp)
P0_hat = 1.0 - P1_hat

P1_flat = P1_hat.ravel()
P0_flat = P0_hat.ravel()

print("\nEstimated CCP  P_hat(a=1 | x, rc_bin):")
print("       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7")
for xi in range(N_X):
    print(f"  x={xi+1}: {np.round(P1_hat[xi], 3)}")

print(f"\nTotal observations recovered from N0+N1: {N_total.sum():.0f}")


TP0 shape: (56, 56)
TP1 shape: (56, 56)
TP0 row sums: [1.00000000, 1.00000000]
TP1 row sums: [1.00000000, 1.00000000]

Estimated CCP  P_hat(a=1 | x, rc_bin):
       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7
  x=1: [0.028 0.019 0.009 0.005 0.004 0.002 0.    0.   ]
  x=2: [0.078 0.066 0.03  0.022 0.015 0.008 0.004 0.008]
  x=3: [0.265 0.173 0.134 0.075 0.049 0.039 0.013 0.011]
  x=4: [0.439 0.343 0.277 0.171 0.128 0.086 0.067 0.025]
  x=5: [0.621 0.531 0.43  0.342 0.252 0.187 0.134 0.082]
  x=6: [0.669 0.63  0.565 0.471 0.369 0.312 0.225 0.091]
  x=7: [0.826 0.74  0.643 0.565 0.478 0.397 0.28  0.208]

Total observations recovered from N0+N1: 100000


### Forward Simulation Approximation of $\hat V(s;\theta)$

For Part (c.1), we follow Equation (5) literally:

$$
\hat V(s;\theta)
=
\frac{1}{N_{\text{sim}}}
\sum_{n=1}^{N_{\text{sim}}}
\sum_{\tau=0}^{T_{\text{sim}}-1}
\beta^\tau
\Bigl[
\bar u(s_{n\tau},a_{n\tau};\theta)
\;+\;
\gamma - \ln \hat P(a_{n\tau}\mid s_{n\tau})
\Bigr].
$$

At each simulated period:

1. Draw an action from the estimated CCP.
2. Draw the next state from the estimated transition matrix conditional on that action.
3. Add the deterministic flow utility plus the Type-I-EV correction term $\gamma - \ln \hat P(a\mid s)$.

The deterministic payoff is

$$
\bar u(s,a;\theta)=
\begin{cases}
-\theta_1 x, & a=0, \\\\
-\theta_2 RC, & a=1.
\end{cases}
$$

Because the CCPs and transition matrices are fixed from the first-stage estimation, we can use common random numbers and simulate all paths once outside the optimizer. Then the simulated value function is linear in $(\theta_1,\theta_2)$:

$$
\hat V(s;\theta)
=
-\theta_1 \cdot M_s
-\theta_2 \cdot R_s
+ K_s,
$$

where

- $M_s$ is the simulated discounted average of $x_\tau \mathbf{1}\{a_\tau=0\}$,
- $R_s$ is the simulated discounted average of $RC_\tau \mathbf{1}\{a_\tau=1\}$,
- $K_s$ is the simulated discounted average of $\gamma - \ln \hat P(a_\tau\mid s_\tau)$.

This keeps the optimizer deterministic and much faster while staying fully faithful to Equation (5).

In [35]:
discounts = BETA ** np.arange(T_SIM)
cumTP0 = np.cumsum(TP0, axis=1)
cumTP1 = np.cumsum(TP1, axis=1)

# Numerical guard: force the last cumulative entry to be exactly 1
cumTP0[:, -1] = 1.0
cumTP1[:, -1] = 1.0


def simulate_value_components(P1_flat, P0_flat, x_of_s, rc_of_s, cumTP0, cumTP1,
                              n_sim=N_SIM, t_sim=T_SIM, seed=SIM_SEED):
    '''
    Pre-simulate the objects in Equation (5) using common random numbers.

    Returns three length-S vectors:
      maint_component[s] = E_sim[sum beta^t * x_t * 1{a_t=0} | s0=s]
      repl_component[s]  = E_sim[sum beta^t * rc_t * 1{a_t=1} | s0=s]
      corr_component[s]  = E_sim[sum beta^t * (gamma - log P_hat(a_t|s_t)) | s0=s]
    '''
    rng = np.random.default_rng(seed)

    maint_component = np.zeros(S)
    repl_component  = np.zeros(S)
    corr_component  = np.zeros(S)

    for s0 in range(S):
        states = np.full(n_sim, s0, dtype=int)

        maint_acc = np.zeros(n_sim)
        repl_acc  = np.zeros(n_sim)
        corr_acc  = np.zeros(n_sim)

        for tau in range(t_sim):
            beta_tau = discounts[tau]

            p1 = P1_flat[states]
            p0 = P0_flat[states]

            u_action = rng.random(n_sim)
            a1 = u_action < p1
            a0 = ~a1

            p_selected = np.where(a1, p1, p0)

            maint_acc += beta_tau * x_of_s[states] * a0
            repl_acc  += beta_tau * rc_of_s[states] * a1
            corr_acc  += beta_tau * (GAMMA - np.log(p_selected))

            u_next = rng.random(n_sim)
            cdf = np.where(a1[:, None], cumTP1[states, :], cumTP0[states, :])
            next_states = (u_next[:, None] > cdf).sum(axis=1)
            next_states = np.minimum(next_states, S - 1)
            states = next_states

        maint_component[s0] = maint_acc.mean()
        repl_component[s0]  = repl_acc.mean()
        corr_component[s0]  = corr_acc.mean()

    return maint_component, repl_component, corr_component


t0_sim = time.time()
maint_component, repl_component, corr_component = simulate_value_components(
    P1_flat=P1_flat,
    P0_flat=P0_flat,
    x_of_s=x_of_s,
    rc_of_s=rc_of_s,
    cumTP0=cumTP0,
    cumTP1=cumTP1,
    n_sim=N_SIM,
    t_sim=T_SIM,
    seed=SIM_SEED,
)
t_simulation = time.time() - t0_sim

print(f"Forward simulation pre-computation finished in {t_simulation:.2f} seconds.")
print(f"maint_component shape: {maint_component.shape}")
print(f"repl_component  shape: {repl_component.shape}")
print(f"corr_component  shape: {corr_component.shape}")

print("\nFirst 10 entries of simulated components:")
print("maint_component:", np.round(maint_component[:10], 3))
print("repl_component :", np.round(repl_component[:10], 3))
print("corr_component :", np.round(corr_component[:10], 3))


Forward simulation pre-computation finished in 0.19 seconds.
maint_component shape: (56,)
repl_component  shape: (56,)
corr_component  shape: (56,)

First 10 entries of simulated components:
maint_component: [50.174 50.543 51.643 51.731 52.566 51.359 51.133 52.636 53.143 52.681]
repl_component : [145.679 141.849 139.79  140.381 139.478 142.39  143.089 140.171 147.691
 149.059]
corr_component : [18.124 17.783 17.523 17.741 17.764 17.752 17.761 17.6   18.406 18.373]


### From Simulated $\hat V(s;\theta)$ to Model CCPs and the Likelihood

Once we have $\hat V(s;\theta)$, we compute the continuation value for each action:

$$
EV(s,0;\theta) = \sum_{s'} TP_0(s,s') \hat V(s';\theta),
\qquad
EV(s,1;\theta) = \sum_{s'} TP_1(s,s') \hat V(s';\theta).
$$

Then the choice-specific values are

$$
\delta_0(s;\theta) = -\theta_1 x_s + \beta EV(s,0;\theta),
$$

$$
\delta_1(s;\theta) = -\theta_2 RC_s + \beta EV(s,1;\theta).
$$

Under Type I EV shocks, the model-implied replacement probability is

$$
\tilde P(a=1\mid s;\theta)
=
\frac{\exp(\delta_1(s;\theta))}
{\exp(\delta_0(s;\theta)) + \exp(\delta_1(s;\theta))}.
$$

Finally, we maximize the same state-level MLE objective as in Parts (a) and (b):

$$
\ell(\theta_1,\theta_2)
=
\sum_{s=1}^{|S|}
\Bigl[
N_1(s)\ln \tilde P(a=1\mid s;\theta)
+
N_0(s)\ln \tilde P(a=0\mid s;\theta)
\Bigr].
$$


In [36]:
def simulated_value(theta1, theta2):
    '''
    Equation (5) value approximation, using the pre-simulated components:

      V_hat(s; theta) = -theta1 * maint_component[s]
                        -theta2 * repl_component[s]
                        + corr_component[s]
    '''
    return -theta1 * maint_component - theta2 * repl_component + corr_component


def model_ccp(theta1, theta2):
    '''
    From V_hat(s;theta), compute continuation values EV(s,a;theta),
    then recover model-implied CCPs from the logit formula.
    '''
    V_hat = simulated_value(theta1, theta2)

    EV0 = TP0 @ V_hat
    EV1 = TP1 @ V_hat

    delta0 = -theta1 * x_of_s  + BETA * EV0
    delta1 = -theta2 * rc_of_s + BETA * EV1

    dmax = np.maximum(delta0, delta1)
    exp0 = np.exp(delta0 - dmax)
    exp1 = np.exp(delta1 - dmax)
    P1_model = exp1 / (exp0 + exp1)

    return V_hat, P1_model.reshape(N_X, N_RC_BINS)


def neg_log_likelihood(params):
    theta1, theta2 = params

    if theta1 <= 0 or theta2 <= 0:
        return 1e10

    _, P1_model = model_ccp(theta1, theta2)
    P0_model = 1.0 - P1_model

    eps = 1e-12
    ll = np.sum(N1 * np.log(P1_model + eps) + N0 * np.log(P0_model + eps))
    return -ll


theta_init = np.array([1.0, 1.0])
V_init, P1_init = model_ccp(theta_init[0], theta_init[1])

print("Initial-guess diagnostics at theta1=1, theta2=1")
print(f"NLL(theta_init) = {neg_log_likelihood(theta_init):.4f}")
print("\nFirst 10 entries of V_hat(theta_init):")
print(np.round(V_init[:10], 3))

print("\nModel-implied P(a=1|x,rc_bin) at theta1=theta2=1:")
print("       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7")
for xi in range(N_X):
    print(f"  x={xi+1}: {np.round(P1_init[xi], 3)}")


Initial-guess diagnostics at theta1=1, theta2=1
NLL(theta_init) = 202947.8948

First 10 entries of V_hat(theta_init):
[-177.729 -174.608 -173.911 -174.371 -174.28  -175.996 -176.461 -175.208
 -182.428 -183.367]

Model-implied P(a=1|x,rc_bin) at theta1=theta2=1:
       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7
  x=1: [0. 0. 0. 0. 0. 0. 0. 0.]
  x=2: [0. 0. 0. 0. 0. 0. 0. 0.]
  x=3: [0. 0. 0. 0. 0. 0. 0. 0.]
  x=4: [0.016 0.    0.    0.    0.    0.    0.    0.   ]
  x=5: [0.069 0.    0.    0.    0.    0.    0.    0.   ]
  x=6: [0.812 0.012 0.005 0.    0.    0.    0.    0.   ]
  x=7: [0.922 0.033 0.013 0.001 0.    0.    0.    0.   ]


### Optimization

The forward simulation has already been done outside the optimizer.
So each optimizer call now does only three things:

1. Construct $\hat V(s;\theta)$ from the simulated components.
2. Compute continuation values and model-implied CCPs.
3. Evaluate the state-level log-likelihood.

For comparability with Parts (a) and (b), we again use Nelder-Mead.

In [37]:
print("--- Part (c.1): Forward Simulation MLE ---")
print(f"Initial theta = {theta_init}")
print(f"Initial NLL   = {neg_log_likelihood(theta_init):.4f}\n")

t0_theta = time.perf_counter()
result = minimize(
    neg_log_likelihood,
    theta_init,
    method="Nelder-Mead",
    options=dict(maxiter=5000, xatol=1e-8, fatol=1e-8, disp=True),
)
optimizer_runtime_sec = time.perf_counter() - t0_theta
optimizer_runtime_ms = 1000.0 * optimizer_runtime_sec
running_time_sec = t_simulation + optimizer_runtime_sec
running_time_ms = 1000.0 * running_time_sec

print(f"\nOptimizer-only time: {optimizer_runtime_sec:.4f} seconds ({optimizer_runtime_ms:.2f} ms)")
print(f"Comparable running time: {running_time_sec:.4f} seconds ({running_time_ms:.2f} ms)")


--- Part (c.1): Forward Simulation MLE ---
Initial theta = [1. 1.]
Initial NLL   = 202947.8948

Optimization terminated successfully.
         Current function value: 34001.052261
         Iterations: 77
         Function evaluations: 151

Optimizer-only time: 0.0045 seconds (4.50 ms)
Comparable running time: 0.1933 seconds (193.34 ms)


### Results


In [38]:
theta1_hat, theta2_hat = result.x
V_hat_final, P1_final = model_ccp(theta1_hat, theta2_hat)

print("=" * 62)
print("     PART (c.1) FORWARD SIMULATION -- MLE RESULTS")
print("=" * 62)
print(f"theta1_hat    (mileage cost coeff.)     = {theta1_hat:.4f}")
print(f"theta2_hat    (replacement cost coeff.) = {theta2_hat:.4f}")
print("-" * 62)
print("Pre-estimated RC process parameters:")
print(f"rho0_hat                                  = {rho0_hat:.4f}")
print(f"rho1_hat                                  = {rho1_hat:.4f}")
print(f"sigma_rho_hat                             = {sigma_rho_hat:.4f}")
print("-" * 62)
print(f"Choice log-likelihood                     = {-result.fun:.4f}")
print(f"Converged                                 = {result.success}")
print(f"Optimizer iterations                      = {result.nit}")
print(f"Forward-simulation precomputation time    = {t_simulation:.4f} sec")
print(f"Optimizer-only time                       = {optimizer_runtime_sec:.4f} sec ({optimizer_runtime_ms:.2f} ms)")
print(f"Comparable running time                   = {running_time_sec:.4f} sec ({running_time_ms:.2f} ms)")
print("=" * 62)

print("\nEstimated model CCP  P(a=1 | x, rc_bin):")
print("       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7")
for xi in range(N_X):
    print(f"  x={xi+1}: {np.round(P1_final[xi], 3)}")


     PART (c.1) FORWARD SIMULATION -- MLE RESULTS
theta1_hat    (mileage cost coeff.)     = 0.3955
theta2_hat    (replacement cost coeff.) = 0.1537
--------------------------------------------------------------
Pre-estimated RC process parameters:
rho0_hat                                  = 17.8269
rho1_hat                                  = 0.6249
sigma_rho_hat                             = 4.6060
--------------------------------------------------------------
Choice log-likelihood                     = -34001.0523
Converged                                 = True
Optimizer iterations                      = 77
Forward-simulation precomputation time    = 0.1888 sec
Optimizer-only time                       = 0.0045 sec (4.50 ms)
Comparable running time                   = 0.1933 sec (193.34 ms)

Estimated model CCP  P(a=1 | x, rc_bin):
       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7
  x=1: [0.028 0.017 0.009 0.006 0.003 0.002 0.001 0.001]
  x=2: [0.091 0.057 0.036 0.026 0.015

In [39]:
master_results['Forward Simulation (Eq. 5)'] = {
    'part': '(c.1)',
    'method': 'Forward Simulation (Eq. 5)',
    'theta1': float(theta1_hat),
    'theta2': float(theta2_hat),
    'running_time_sec': float(running_time_sec),
    'running_time_ms': float(running_time_ms),
}
master_common['Forward Simulation (Eq. 5)'] = {
    'rho0': float(rho0_hat),
    'rho1': float(rho1_hat),
    'sigma_rho': float(sigma_rho_hat),
}
print('Saved summary for Part (c.1) in master_results.')


Saved summary for Part (c.1) in master_results.


### Summary

The first two steps are unchanged from Part (b):

1. Estimate the RC transition objects and the AR(1) parameters.
2. Estimate the CCP from the data.

The only change is Step 3:

- Part (b): use the Hotz-Miller linear inversion $AV=b(\theta)$.
- Part (c.1): use forward simulation and Equation (5) to approximate $\hat V(s;\theta)$.

Then, just as before, we:

1. Compute $EV(s,a;\theta)$ from the transition matrix and $\hat V$.
2. Recover model-implied choice probabilities from the logit formula.
3. Form the state-level MLE objective.


---
## Part (c.2): Forward Simulation Using Equations (6) and (7)


# ECON 662 — Problem Set 2
## Part (c.2): Forward Simulation MLE Using Equations (6) and (7)

Estimate $(\theta,\rho,\sigma_\rho)$ using the **second forward-simulation approach** in the homework notes.

The problem set says:

- the first two steps are the same as Part (b),
- for Part (c.2), use Equation (6) and Equation (7),
- then compute continuation values, form model CCPs, and maximize the likelihood.

We also follow the methodological structure from Lecture 15:

1. Estimate transitions.
2. Estimate CCPs from the data.
3. For each guess of $\theta$, approximate the value function by forward simulation.
4. Use the approximated value function to form model CCPs and then compute the MLE objective.


### Import Libraries


In [40]:
import time
import numpy as np
import pandas as pd
from scipy.optimize import minimize

np.random.seed(42)


### Load Data


In [41]:
# Variables:
# i  = bus id
# t  = month
# a  = action (0 = maintain, 1 = replace)
# x  = mileage state in {1, ..., 7}
# rc = replacement cost (continuous)
df = pd.read_csv("ddc_pset.csv")
df = df.sort_values(["i", "t"]).reset_index(drop=True)

print(f"Loaded {len(df):,} observations for {df['i'].nunique():,} buses.")
print(f"RC range: [{df['rc'].min():.4f}, {df['rc'].max():.4f}]")
print(f"Replacement rate: {df['a'].mean():.4f}")
df.head(10)


Loaded 100,000 observations for 1,000 buses.
RC range: [31.5000, 62.5000]
Replacement rate: 0.1724


,i,t,a,x,rc
0,1,1,0,4,45.0
1,1,2,0,5,49.0
2,1,3,0,6,54.0
3,1,4,1,7,47.0
4,1,5,0,1,56.0
5,1,6,0,2,52.5
6,1,7,0,3,55.0
7,1,8,0,4,50.5
8,1,9,0,5,52.0
9,1,10,0,6,42.0


### Global Constants


In [42]:
BETA      = 0.95
N_X       = 7
N_RC_BINS = 8
S         = N_X * N_RC_BINS

# Forward-simulation tuning choices for Part (c.2)
N_SIM     = 100
T_SIM     = 120
SIM_SEED  = 24680

print(f"beta        = {BETA}")
print(f"N_X         = {N_X}")
print(f"N_RC_BINS   = {N_RC_BINS}")
print(f"|S|         = {S}")
print(f"N_SIM       = {N_SIM}")
print(f"T_SIM       = {T_SIM}")
print(f"beta^T_SIM  = {BETA**T_SIM:.6f}")


beta        = 0.95
N_X         = 7
N_RC_BINS   = 8
|S|         = 56
N_SIM       = 100
T_SIM       = 120
beta^T_SIM  = 0.002122


### Estimate the RC Process and Build the State Grid

As in Parts (b) and (c.1), we keep the first-step objects fixed throughout the MLE:

1. Build an empirical RC grid with equal-width bins.
2. Estimate the RC transition matrix $\Pi$ directly from observed bin-to-bin movements.
3. Estimate the continuous AR(1) process

$$
RC_t = \rho_0 + \rho_1 RC_{t-1} + e_t,\qquad e_t \sim N(0,\sigma_\rho^2),
$$
by OLS.


In [43]:
# Equal-width RC bins from the observed support
rc_min = df['rc'].min()
rc_max = df['rc'].max()
bin_edges = np.linspace(rc_min, rc_max, N_RC_BINS + 1)
rc_grid = (bin_edges[:-1] + bin_edges[1:]) / 2.0

# Assign each observation to a bin (0-indexed)
rc_bin_idx = np.searchsorted(bin_edges[1:], df['rc'].values, side='left')
rc_bin_idx = np.clip(rc_bin_idx, 0, N_RC_BINS - 1)
df['rc_bin'] = rc_bin_idx
df['rc_mid'] = rc_grid[rc_bin_idx]

print("RC bin edges:")
print(np.round(bin_edges, 3))
print("\nRC grid (bin midpoints):")
print(np.round(rc_grid, 3))

# Empirical RC transition matrix Pi
C = np.zeros((N_RC_BINS, N_RC_BINS))
for _, g in df.groupby("i"):
    bins = g['rc_bin'].values
    for j_from, j_to in zip(bins[:-1], bins[1:]):
        C[j_from, j_to] += 1

Pi = C / C.sum(axis=1, keepdims=True)

print("\nEmpirical RC transition matrix Pi:")
print(np.round(Pi, 4))
print("Row sums:")
print(np.round(Pi.sum(axis=1), 10))

# OLS for the continuous AR(1) process
rc_prev_list, rc_curr_list = [], []
for _, g in df.groupby("i"):
    vals = g['rc'].values
    rc_prev_list.append(vals[:-1])
    rc_curr_list.append(vals[1:])

rc_prev = np.concatenate(rc_prev_list)
rc_curr = np.concatenate(rc_curr_list)

X_ols = np.column_stack([np.ones(len(rc_prev)), rc_prev])
beta_ols = np.linalg.solve(X_ols.T @ X_ols, X_ols.T @ rc_curr)
rho0_hat, rho1_hat = beta_ols
sigma_rho_hat = (rc_curr - X_ols @ beta_ols).std(ddof=1)

print("\nAR(1) estimates from continuous RC:")
print(f"rho0_hat      = {rho0_hat:.4f}")
print(f"rho1_hat      = {rho1_hat:.4f}")
print(f"sigma_rho_hat = {sigma_rho_hat:.4f}")


RC bin edges:
[31.5   35.375 39.25  43.125 47.    50.875 54.75  58.625 62.5  ]

RC grid (bin midpoints):
[33.438 37.312 41.188 45.062 48.938 52.812 56.688 60.562]

Empirical RC transition matrix Pi:
[[0.     0.3333 0.6667 0.     0.     0.     0.     0.    ]
 [0.2    0.4    0.2    0.2    0.     0.     0.     0.    ]
 [0.1333 0.     0.4    0.2    0.2    0.0667 0.     0.    ]
 [0.     0.087  0.087  0.3043 0.3913 0.087  0.0435 0.    ]
 [0.     0.     0.1154 0.2692 0.2692 0.3077 0.0385 0.    ]
 [0.     0.     0.0556 0.2778 0.2778 0.1111 0.2222 0.0556]
 [0.     0.     0.     0.     0.2857 0.5714 0.     0.1429]
 [0.     0.     0.     0.     0.     0.5    0.5    0.    ]]
Row sums:
[1. 1. 1. 1. 1. 1. 1. 1.]

AR(1) estimates from continuous RC:
rho0_hat      = 17.8269
rho1_hat      = 0.6249
sigma_rho_hat = 4.6060


### Build the Full Transition Matrices and Estimate CCPs

The combined state is $s=(x,j)$, where:

- $x \in \{1,\dots,7\}$ is mileage,
- $j \in \{0,\dots,7\}$ is the RC bin.

Hence the model has

$$
|S| = 7 \times 8 = 56
$$

discrete states.

The state transitions are:

- if $a=0$ (maintain), then $x'=\min(x+1,7)$,
- if $a=1$ (replace), then $x'=1$,
- and the RC-bin transition is given by the empirical matrix $\Pi$.

We also estimate the conditional choice probabilities using state frequencies:

$$
\hat P(a=1\mid s)=\frac{N_1(s)}{N_0(s)+N_1(s)},
\qquad
\hat P(a=0\mid s)=1-\hat P(a=1\mid s).
$$

As in Part (c.1), we clip the estimated CCPs slightly away from 0 and 1 so the log-odds are always finite.


In [44]:
# Flat state indexing: s = xi * N_RC_BINS + j
xi_of_s = np.array([xi for xi in range(N_X) for _ in range(N_RC_BINS)])
j_of_s  = np.array([j  for _ in range(N_X) for j in range(N_RC_BINS)])
x_of_s  = xi_of_s + 1
rc_of_s = rc_grid[j_of_s]

# Next mileage under each action
xi_next_mnt = np.minimum(xi_of_s + 1, N_X - 1)
xi_next_rep = np.zeros(S, dtype=int)

# Action-specific transition matrices TP0 and TP1
TP0 = np.zeros((S, S))
TP1 = np.zeros((S, S))
for s in range(S):
    j = j_of_s[s]

    col0 = xi_next_mnt[s] * N_RC_BINS
    TP0[s, col0:col0 + N_RC_BINS] = Pi[j, :]

    col1 = xi_next_rep[s] * N_RC_BINS
    TP1[s, col1:col1 + N_RC_BINS] = Pi[j, :]

print(f"TP0 shape: {TP0.shape}")
print(f"TP1 shape: {TP1.shape}")
print(f"TP0 row sums: [{TP0.sum(1).min():.8f}, {TP0.sum(1).max():.8f}]")
print(f"TP1 row sums: [{TP1.sum(1).min():.8f}, {TP1.sum(1).max():.8f}]")

# State-level counts
N1 = np.zeros((N_X, N_RC_BINS))
N0 = np.zeros((N_X, N_RC_BINS))

for xi_obs, j_obs, a_obs in zip(df['x'].values.astype(int) - 1,
                                df['rc_bin'].values.astype(int),
                                df['a'].values.astype(int)):
    if a_obs == 1:
        N1[xi_obs, j_obs] += 1
    else:
        N0[xi_obs, j_obs] += 1

N_total = N0 + N1

eps_ccp = 1e-6
P1_hat = np.where(N_total > 0, N1 / N_total, eps_ccp)
P1_hat = np.clip(P1_hat, eps_ccp, 1.0 - eps_ccp)
P0_hat = 1.0 - P1_hat

P1_flat = P1_hat.ravel()
P0_flat = P0_hat.ravel()
log_odds_flat = np.log(P1_flat) - np.log(P0_flat)

print("\nEstimated CCP  P_hat(a=1 | x, rc_bin):")
print("       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7")
for xi in range(N_X):
    print(f"  x={xi+1}: {np.round(P1_hat[xi], 3)}")

print(f"\nTotal observations recovered from N0+N1: {N_total.sum():.0f}")


TP0 shape: (56, 56)
TP1 shape: (56, 56)
TP0 row sums: [1.00000000, 1.00000000]
TP1 row sums: [1.00000000, 1.00000000]

Estimated CCP  P_hat(a=1 | x, rc_bin):
       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7
  x=1: [0.028 0.019 0.009 0.005 0.004 0.002 0.    0.   ]
  x=2: [0.078 0.066 0.03  0.022 0.015 0.008 0.004 0.008]
  x=3: [0.265 0.173 0.134 0.075 0.049 0.039 0.013 0.011]
  x=4: [0.439 0.343 0.277 0.171 0.128 0.086 0.067 0.025]
  x=5: [0.621 0.531 0.43  0.342 0.252 0.187 0.134 0.082]
  x=6: [0.669 0.63  0.565 0.471 0.369 0.312 0.225 0.091]
  x=7: [0.826 0.74  0.643 0.565 0.478 0.397 0.28  0.208]

Total observations recovered from N0+N1: 100000


### Proof of Equation (6)

Let the deterministic choice-specific value be

$$
\delta_0(s;\theta)=\bar u(s,0;\theta)+\beta EV(s,0;\theta),
\qquad
\delta_1(s;\theta)=\bar u(s,1;\theta)+\beta EV(s,1;\theta).
$$

The agent chooses action 1 if and only if

$$
\delta_1(s;\theta)+\varepsilon_1 > \delta_0(s;\theta)+\varepsilon_0.
$$

Rearranging:

$$
\delta_1(s;\theta)-\delta_0(s;\theta)+\varepsilon_1-\varepsilon_0 > 0.
$$

Under Type I EV shocks, the corresponding logit CCP is

$$
P(a=1\mid s)=
\frac{\exp(\delta_1(s;\theta))}
{\exp(\delta_0(s;\theta))+\exp(\delta_1(s;\theta))},
\qquad
P(a=0\mid s)=
\frac{\exp(\delta_0(s;\theta))}
{\exp(\delta_0(s;\theta))+\exp(\delta_1(s;\theta))}.
$$

Therefore

$$
\frac{P(a=1\mid s)}{P(a=0\mid s)}
=
\exp\bigl(\delta_1(s;\theta)-\delta_0(s;\theta)\bigr),
$$

so

$$
\delta_1(s;\theta)-\delta_0(s;\theta)
=
\ln\!\left(\frac{P(a=1\mid s)}{P(a=0\mid s)}\right).
$$

Substituting this back into the choice rule gives

$$
\text{action}=1
\iff
\ln\!\left(\frac{P(a=1\mid s)}{P(a=0\mid s)}\right)
+ \varepsilon_1-\varepsilon_0 > 0,
$$

which is exactly Equation (6). In the simulation code we replace the true CCP by the first-step estimate $\hat P$, giving

$$
\text{action}=1
\iff
\ln\!\left(\frac{\hat P(a=1\mid s)}{\hat P(a=0\mid s)}\right)
+ \varepsilon_1-\varepsilon_0 > 0.
$$

Because the event of exact equality has probability zero under continuous shocks, using $>$ versus $\geq$ is trivial.


### Forward Simulation Using Direct Shock Draws

Now we implement the Part (c.2) simulation step exactly as in Equations (6) and (7).

At each simulated period and current state $s_\tau$:

1. Draw
   $$
   \varepsilon_{\tau,0}, \varepsilon_{\tau,1}
   \sim \text{Type I Extreme Value}
   $$
   independently.
2. Choose the action using Equation (6):
   $$
   a_\tau = 1
   \iff
   \ln\!\left(\frac{\hat P(a=1\mid s_\tau)}{\hat P(a=0\mid s_\tau)}\right)
   + \varepsilon_{\tau,1} - \varepsilon_{\tau,0} > 0.
   $$
3. Transition to the next state using the estimated transition matrix conditional on the chosen action.
4. Add the simulated payoff from Equation (7):
   $$
   \sum_{\tau=0}^{T_{\text{sim}}-1}
   \beta^\tau
   \left[
   \bar u(s_\tau,a_\tau;\theta) + \varepsilon_{\tau,a_\tau}
   \right].
   $$

Once again, because the estimated CCPs are fixed before the optimizer runs, the whole simulated path is independent of $\theta$ once the shocks are drawn. That means we can pre-compute:

$$
\hat V(s;\theta)
=
-\theta_1 M_s
-\theta_2 R_s
+ E_s,
$$

where

- $M_s$ is the simulated discounted mileage cost component,
- $R_s$ is the simulated discounted replacement-cost component,
- $E_s$ is the simulated discounted selected-shock component $\varepsilon_{\tau,a_\tau}$.


In [45]:
discounts = BETA ** np.arange(T_SIM)
cumTP0 = np.cumsum(TP0, axis=1)
cumTP1 = np.cumsum(TP1, axis=1)
cumTP0[:, -1] = 1.0
cumTP1[:, -1] = 1.0


def simulate_value_components_eq67():
    # Pre-compute the simulation objects in Equation (7) using common random numbers.
    rng = np.random.default_rng(SIM_SEED)

    maint_component = np.zeros(S)
    repl_component = np.zeros(S)
    shock_component = np.zeros(S)

    for s0 in range(S):
        states = np.full(N_SIM, s0, dtype=int)

        maint_acc = np.zeros(N_SIM)
        repl_acc = np.zeros(N_SIM)
        shock_acc = np.zeros(N_SIM)

        for tau in range(T_SIM):
            beta_tau = discounts[tau]

            eps0 = rng.gumbel(loc=0.0, scale=1.0, size=N_SIM)
            eps1 = rng.gumbel(loc=0.0, scale=1.0, size=N_SIM)

            log_odds = log_odds_flat[states]
            a1 = (log_odds + eps1 - eps0) > 0.0
            a0 = ~a1

            maint_acc += beta_tau * x_of_s[states] * a0
            repl_acc += beta_tau * rc_of_s[states] * a1

            selected_eps = np.where(a1, eps1, eps0)
            shock_acc += beta_tau * selected_eps

            u_next = rng.random(N_SIM)
            cdf = np.where(a1[:, None], cumTP1[states, :], cumTP0[states, :])
            next_states = (u_next[:, None] > cdf).sum(axis=1)
            next_states = np.minimum(next_states, S - 1)
            states = next_states

        maint_component[s0] = maint_acc.mean()
        repl_component[s0] = repl_acc.mean()
        shock_component[s0] = shock_acc.mean()

    return maint_component, repl_component, shock_component


t0_sim = time.time()
maint_component, repl_component, shock_component = simulate_value_components_eq67()
t_simulation = time.time() - t0_sim

print(f"Forward simulation pre-computation finished in {t_simulation:.2f} seconds.")
print(f"maint_component shape: {maint_component.shape}")
print(f"repl_component  shape: {repl_component.shape}")
print(f"shock_component shape: {shock_component.shape}")

print("\nFirst 10 entries of simulated components:")
print("maint_component:", np.round(maint_component[:10], 3))
print("repl_component :", np.round(repl_component[:10], 3))
print("shock_component:", np.round(shock_component[:10], 3))


Forward simulation pre-computation finished in 0.27 seconds.
maint_component shape: (56,)
repl_component  shape: (56,)
shock_component shape: (56,)

First 10 entries of simulated components:
maint_component: [51.079 51.43  51.076 52.839 52.833 51.478 51.713 52.395 51.552 51.604]
repl_component : [142.25  140.857 142.346 138.855 139.003 141.891 142.755 139.037 151.412
 150.338]
shock_component: [17.921 17.905 17.314 17.682 18.265 17.944 17.211 16.924 18.303 19.401]


### From Simulated $\hat V(s;\theta)$ to Model CCPs and the Likelihood

After simulating the value function, we compute action-specific continuation values:

$$
EV(s,0;\theta)=\sum_{s'} TP_0(s,s')\hat V(s';\theta),
\qquad
EV(s,1;\theta)=\sum_{s'} TP_1(s,s')\hat V(s';\theta).
$$

The deterministic choice-specific values are then

$$
\delta_0(s;\theta)=-\theta_1 x_s + \beta EV(s,0;\theta),
\qquad
\delta_1(s;\theta)=-\theta_2 RC_s + \beta EV(s,1;\theta).
$$

Under the logit formula, the model-implied CCP is

$$
\tilde P(a=1\mid s;\theta)
=
\frac{\exp(\delta_1(s;\theta))}
{\exp(\delta_0(s;\theta))+\exp(\delta_1(s;\theta))}.
$$

Finally we maximize the same state-level MLE objective as in Parts (a), (b), and (c.1):

$$
\ell(\theta_1,\theta_2)
=
\sum_{s=1}^{|S|}
\left[
N_1(s)\ln \tilde P(a=1\mid s;\theta)
+
N_0(s)\ln \tilde P(a=0\mid s;\theta)
\right].
$$


In [46]:
def simulated_value(theta1, theta2):
    # Equation (7) approximation:
    # V_hat(s;theta) = -theta1 * maint_component
    #                  -theta2 * repl_component
    #                  +shock_component
    return -theta1 * maint_component - theta2 * repl_component + shock_component


def model_ccp(theta1, theta2):
    V_hat = simulated_value(theta1, theta2)

    EV0 = TP0 @ V_hat
    EV1 = TP1 @ V_hat

    delta0 = -theta1 * x_of_s + BETA * EV0
    delta1 = -theta2 * rc_of_s + BETA * EV1

    dmax = np.maximum(delta0, delta1)
    exp0 = np.exp(delta0 - dmax)
    exp1 = np.exp(delta1 - dmax)
    P1_model = exp1 / (exp0 + exp1)

    return V_hat, P1_model.reshape(N_X, N_RC_BINS)


def neg_log_likelihood(params):
    theta1, theta2 = params

    if theta1 <= 0 or theta2 <= 0:
        return 1e10

    _, P1_model = model_ccp(theta1, theta2)
    P0_model = 1.0 - P1_model

    eps = 1e-12
    ll = np.sum(N1 * np.log(P1_model + eps) + N0 * np.log(P0_model + eps))
    return -ll


theta_init = np.array([1.0, 1.0])
V_init, P1_init = model_ccp(theta_init[0], theta_init[1])

print("Initial-guess diagnostics at theta1=1, theta2=1")
print(f"NLL(theta_init) = {neg_log_likelihood(theta_init):.4f}")
print("\nFirst 10 entries of V_hat(theta_init):")
print(np.round(V_init[:10], 3))

print("\nModel-implied P(a=1|x,rc_bin) at theta1=theta2=1:")
print("       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7")
for xi in range(N_X):
    print(f"  x={xi+1}: {np.round(P1_init[xi], 3)}")


Initial-guess diagnostics at theta1=1, theta2=1
NLL(theta_init) = 197218.8643

First 10 entries of V_hat(theta_init):
[-175.408 -174.382 -176.108 -174.012 -173.572 -175.425 -177.257 -174.509
 -184.662 -182.54 ]

Model-implied P(a=1|x,rc_bin) at theta1=theta2=1:
       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7
  x=1: [0. 0. 0. 0. 0. 0. 0. 0.]
  x=2: [0. 0. 0. 0. 0. 0. 0. 0.]
  x=3: [0.001 0.    0.    0.    0.    0.    0.    0.   ]
  x=4: [0.004 0.    0.    0.    0.    0.    0.    0.   ]
  x=5: [0.302 0.002 0.001 0.    0.    0.    0.    0.   ]
  x=6: [0.27  0.009 0.002 0.    0.    0.    0.    0.   ]
  x=7: [0.502 0.024 0.004 0.001 0.    0.    0.    0.   ]


### Optimization

For comparability with the earlier parts, we again use Nelder-Mead.

The optimizer now treats the first-step objects, the simulated shock draws, and the simulated paths as fixed. Each optimizer call only:

1. forms $\hat V(s;\theta)$,
2. computes continuation values and model CCPs,
3. evaluates the state-level log-likelihood.


In [47]:
print("--- Part (c.2): Forward Simulation with Direct Shock Draws ---")
print(f"Initial theta = {theta_init}")
print(f"Initial NLL   = {neg_log_likelihood(theta_init):.4f}\n")

t0_theta = time.perf_counter()
result = minimize(
    neg_log_likelihood,
    theta_init,
    method="Nelder-Mead",
    options=dict(maxiter=5000, xatol=1e-8, fatol=1e-8, disp=True),
)
optimizer_runtime_sec = time.perf_counter() - t0_theta
optimizer_runtime_ms = 1000.0 * optimizer_runtime_sec
running_time_sec = t_simulation + optimizer_runtime_sec
running_time_ms = 1000.0 * running_time_sec

print(f"\nOptimizer-only time: {optimizer_runtime_sec:.4f} seconds ({optimizer_runtime_ms:.2f} ms)")
print(f"Comparable running time: {running_time_sec:.4f} seconds ({running_time_ms:.2f} ms)")


--- Part (c.2): Forward Simulation with Direct Shock Draws ---
Initial theta = [1. 1.]
Initial NLL   = 197218.8643

Optimization terminated successfully.
         Current function value: 34408.221935
         Iterations: 81
         Function evaluations: 159

Optimizer-only time: 0.0049 seconds (4.86 ms)
Comparable running time: 0.2703 seconds (270.28 ms)


### Results


In [48]:
theta1_hat, theta2_hat = result.x
V_hat_final, P1_final = model_ccp(theta1_hat, theta2_hat)

print("=" * 64)
print("     PART (c.2) FORWARD SIMULATION -- DIRECT SHOCK DRAWS")
print("=" * 64)
print(f"theta1_hat    (mileage cost coeff.)     = {theta1_hat:.4f}")
print(f"theta2_hat    (replacement cost coeff.) = {theta2_hat:.4f}")
print("-" * 64)
print("Pre-estimated RC process parameters:")
print(f"rho0_hat                                  = {rho0_hat:.4f}")
print(f"rho1_hat                                  = {rho1_hat:.4f}")
print(f"sigma_rho_hat                             = {sigma_rho_hat:.4f}")
print("-" * 64)
print(f"Choice log-likelihood                     = {-result.fun:.4f}")
print(f"Converged                                 = {result.success}")
print(f"Optimizer iterations                      = {result.nit}")
print(f"Forward-simulation precomputation time    = {t_simulation:.4f} sec")
print(f"Optimizer-only time                       = {optimizer_runtime_sec:.4f} sec ({optimizer_runtime_ms:.2f} ms)")
print(f"Comparable running time                   = {running_time_sec:.4f} sec ({running_time_ms:.2f} ms)")
print("=" * 64)

print("\nEstimated model CCP  P(a=1 | x, rc_bin):")
print("       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7")
for xi in range(N_X):
    print(f"  x={xi+1}: {np.round(P1_final[xi], 3)}")


     PART (c.2) FORWARD SIMULATION -- DIRECT SHOCK DRAWS
theta1_hat    (mileage cost coeff.)     = 0.3897
theta2_hat    (replacement cost coeff.) = 0.1537
----------------------------------------------------------------
Pre-estimated RC process parameters:
rho0_hat                                  = 17.8269
rho1_hat                                  = 0.6249
sigma_rho_hat                             = 4.6060
----------------------------------------------------------------
Choice log-likelihood                     = -34408.2219
Converged                                 = True
Optimizer iterations                      = 81
Forward-simulation precomputation time    = 0.2654 sec
Optimizer-only time                       = 0.0049 sec (4.86 ms)
Comparable running time                   = 0.2703 sec (270.28 ms)

Estimated model CCP  P(a=1 | x, rc_bin):
       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7
  x=1: [0.02  0.01  0.01  0.006 0.003 0.002 0.001 0.   ]
  x=2: [0.109 0.054 0.058 

In [49]:
master_results['Forward Simulation (Eq. 6)-(7)'] = {
    'part': '(c.2)',
    'method': 'Forward Simulation (Eq. 6)-(7)',
    'theta1': float(theta1_hat),
    'theta2': float(theta2_hat),
    'running_time_sec': float(running_time_sec),
    'running_time_ms': float(running_time_ms),
}
master_common['Forward Simulation (Eq. 6)-(7)'] = {
    'rho0': float(rho0_hat),
    'rho1': float(rho1_hat),
    'sigma_rho': float(sigma_rho_hat),
}
print('Saved summary for Part (c.2) in master_results.')


Saved summary for Part (c.2) in master_results.


### Summary

The first two steps are the same as in Parts (b) and (c.1):

1. Estimate the RC transition process and discretize RC with empirical bins.
2. Estimate the CCP from the data.

The difference is the simulation step:

- Part (c.1): draw actions from $\hat P(a\mid s)$ and add the correction term $\gamma - \ln \hat P(a\mid s)$.
- Part (c.2): draw the Type I EV shocks directly, use Equation (6) to choose the action, and add the realized selected shock $\varepsilon_a$ as in Equation (7).


---
## Part (d): Scott (2014) Linear Regression


# ECON 662 — Problem Set 2
## Part (d): Linear Regression Approach (Scott 2014)

Estimate $\theta$ without solving the value function in the estimation algorithm and without estimating the replacement-cost process explicitly, using the linear-regression approach discussed in Lecture 15.

This notebook follows the logic of Scott (2014) as adapted to this bus-replacement problem:

1. Use observed data to estimate conditional choice probabilities (CCPs).
2. Use a termination-action argument to turn future value differences into observable CCP differences.
3. Rearrange the model into a linear estimating equation in $(\theta_1,\theta_2)$.
4. Estimate $(\theta_1,\theta_2)$ by OLS.

#### Import Libraries


In [50]:
import time
import numpy as np
import pandas as pd

np.random.seed(42)


#### Load Data


In [51]:
# Variables:
# i  = bus id
# t  = month
# a  = action (0 = maintain, 1 = replace)
# x  = mileage state in {1, ..., 7}
# rc = replacement cost
df = pd.read_csv("ddc_pset.csv")
df = df.sort_values(["i", "t"]).reset_index(drop=True)

print(f"Loaded {len(df):,} observations for {df['i'].nunique():,} buses.")
print(f"Months in panel: {df['t'].min()} to {df['t'].max()}")
print(f"Mileage states: {sorted(df['x'].unique().tolist())}")
print(f"Replacement rate: {df['a'].mean():.4f}")
df.head(10)


Loaded 100,000 observations for 1,000 buses.
Months in panel: 1 to 100
Mileage states: [1, 2, 3, 4, 5, 6, 7]
Replacement rate: 0.1724


,i,t,a,x,rc
0,1,1,0,4,45.0
1,1,2,0,5,49.0
2,1,3,0,6,54.0
3,1,4,1,7,47.0
4,1,5,0,1,56.0
5,1,6,0,2,52.5
6,1,7,0,3,55.0
7,1,8,0,4,50.5
8,1,9,0,5,52.0
9,1,10,0,6,42.0


#### Global Constants


In [52]:
BETA = 0.95
N_X = 7
T_MAX = int(df["t"].max())

print(f"beta  = {BETA}")
print(f"N_X   = {N_X}")
print(f"T_MAX = {T_MAX}")


beta  = 0.95
N_X   = 7
T_MAX = 100


#### Usual discretization

For this simulated dataset, instead of discretizing and estimating an RC law of motion, we work directly with the observed time path (as usual)

In [53]:
# Check that RC_t is common across buses within each month
rc_counts_by_t = df.groupby("t")["rc"].nunique()
assert rc_counts_by_t.eq(1).all(), "RC must be common across buses within each month for this approach."

rc_by_t = df.groupby("t")["rc"].first().to_numpy()   # length 100

print("Unique RC values per t (should all equal 1):")
print(rc_counts_by_t.describe())
print(f"All equal to 1?  {rc_counts_by_t.eq(1).all()}")

print("\nFirst 15 values of the realized RC path:")
print(np.round(rc_by_t[:15], 3))


Unique RC values per t (should all equal 1):
count    100.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: rc, dtype: float64
All equal to 1?  True

First 15 values of the realized RC path:
[45.  49.  54.  47.  56.  52.5 55.  50.5 52.  42.  31.5 39.5 40.5 39.5
 42. ]


### Estimate $\hat P_t(a=1\mid x)$ as a Flexible Function of $x$

The homework note for part (d) says:

> For (d), you can estimate CCPs for each $t$ as a flexible function of $x$.

We follow that literally. Since $RC_t$ is common within a month, the relevant CCP for this approach is

$$
\hat P_t(a=1\mid x),
$$

estimated separately for each month $t$ as a flexible function of mileage $x$.

To keep logs well-defined and avoid zero-probability cells, we use a cubic logit smoother for each month:

$$
\Pr(a=1\mid x,t) = \Lambda(\alpha_{0t} + \alpha_{1t}x + \alpha_{2t}x^2 + \alpha_{3t}x^3),
$$

where

$$
\Lambda(z)=\frac{1}{1+e^{-z}}.
$$

We estimate each month's coefficient vector by Newton-Raphson / IRLS using explicit matrix algebra:

$$
g(\alpha_t)=Z_t^\top(a_t - p_t), \qquad
H(\alpha_t)=-Z_t^\top W_t Z_t,
$$

and update

$$
\alpha_t^{new} = \alpha_t - H(\alpha_t)^{-1} g(\alpha_t).
$$


It gives smoother CCPs than raw frequencies.

In [54]:
def fit_logit_cubic(x, a, maxiter=100, tol=1e-10):
    '''
    Fit logit Pr(a=1|x) = logistic(alpha0 + alpha1*x + alpha2*x^2 + alpha3*x^3)
    using Newton-Raphson / IRLS with explicit matrix algebra.

    Parameters
    ----------
    x : ndarray (n,)
    a : ndarray (n,)

    Returns
    -------
    alpha : ndarray (4,)
    '''
    Z = np.column_stack([
        np.ones_like(x, dtype=float),
        x.astype(float),
        x.astype(float) ** 2,
        x.astype(float) ** 3,
    ])  # shape (n, 4)

    alpha = np.zeros(Z.shape[1])

    for _ in range(maxiter):
        z = Z @ alpha
        z = np.clip(z, -30.0, 30.0)
        p = 1.0 / (1.0 + np.exp(-z))

        # IRLS objects
        w = np.clip(p * (1.0 - p), 1e-8, None)
        grad = Z.T @ (a - p)
        hess = -(Z.T * w) @ Z

        step = np.linalg.solve(hess, grad)
        alpha_new = alpha - step

        if np.max(np.abs(alpha_new - alpha)) < tol:
            alpha = alpha_new
            break
        alpha = alpha_new

    return alpha


t0_method = time.perf_counter()

# Time-by-mileage count matrices
N1_tx = np.zeros((T_MAX, N_X))
N0_tx = np.zeros((T_MAX, N_X))

for t, x, a in zip(df["t"].values.astype(int),
                   df["x"].values.astype(int),
                   df["a"].values.astype(int)):
    if a == 1:
        N1_tx[t - 1, x - 1] += 1
    else:
        N0_tx[t - 1, x - 1] += 1

N_tx = N0_tx + N1_tx
freq_p1_tx = np.where(N_tx > 0, N1_tx / N_tx, np.nan)

# Smoothed CCP matrix P1_hat_tx[t, x-1]
P1_hat_tx = np.zeros((T_MAX, N_X))
alpha_by_t = np.zeros((T_MAX, 4))
x_grid = np.arange(1, N_X + 1, dtype=float)
Z_grid = np.column_stack([
    np.ones(N_X),
    x_grid,
    x_grid ** 2,
    x_grid ** 3,
])  # shape (7, 4)

for t in range(1, T_MAX + 1):
    g = df.loc[df["t"] == t, ["x", "a"]]
    x_t = g["x"].to_numpy(dtype=float)
    a_t = g["a"].to_numpy(dtype=float)

    alpha_t = fit_logit_cubic(x_t, a_t)
    alpha_by_t[t - 1, :] = alpha_t

    p_t = 1.0 / (1.0 + np.exp(-np.clip(Z_grid @ alpha_t, -30.0, 30.0)))
    P1_hat_tx[t - 1, :] = np.clip(p_t, 1e-6, 1.0 - 1e-6)

P0_hat_tx = 1.0 - P1_hat_tx

print("Time-by-mileage count matrix N_tx shape:", N_tx.shape)
print("Smoothed CCP matrix P1_hat_tx shape:", P1_hat_tx.shape)

print("\nFirst 3 months: raw frequency CCP P(a=1|x,t)")
print(np.round(freq_p1_tx[:3], 3))

print("\nFirst 3 months: smoothed CCP P_hat(a=1|x,t)")
print(np.round(P1_hat_tx[:3], 3))


Time-by-mileage count matrix N_tx shape: (100, 7)
Smoothed CCP matrix P1_hat_tx shape: (100, 7)

First 3 months: raw frequency CCP P(a=1|x,t)
[[0.004 0.012 0.093 0.158 0.312 0.423 0.558]
 [0.    0.013 0.072 0.131 0.229 0.412 0.52 ]
 [0.    0.    0.03  0.065 0.144 0.365 0.484]]

First 3 months: smoothed CCP P_hat(a=1|x,t)
[[0.003 0.021 0.078 0.181 0.303 0.426 0.559]
 [0.002 0.014 0.058 0.142 0.252 0.377 0.536]
 [0.001 0.004 0.021 0.069 0.166 0.313 0.497]]


### Derive the Scott (2014) Linear Regression Equation

Let the current state be $(x_t, RC_t)$.

Under Type I EV errors, we can use the identity

$$
V(s) = -\ln P(a=1\mid s) + \delta_1(s) + \gamma,
$$

for any action label $1$, where

$$
\delta_1(s) = \bar u(s,1;\theta) + \beta EV(s,1;\theta).
$$

For the bus problem, action $1$ (replace) is the key "termination-action-style" choice because after replacing, the mileage next period resets to 1. Therefore,

$$
\delta_1(x,RC_t;\theta)
=
-\theta_2 RC_t + \beta V(1, RC_{t+1}),
$$

which does not depend on current mileage $x$ except through the common $RC_t$.

Now start from the current-period log-odds:

$$
\ln\!\left(\frac{P(a=0\mid x_t,RC_t)}{P(a=1\mid x_t,RC_t)}\right)
=
\delta_0(x_t,RC_t;\theta)-\delta_1(x_t,RC_t;\theta).
$$

Using the model payoffs,

$$
\delta_0(x_t,RC_t;\theta)
=
-\theta_1 x_t + \beta V(\min(x_t+1,7), RC_{t+1}),
$$

so

$$
\ln\!\left(\frac{P(a=0\mid x_t,RC_t)}{P(a=1\mid x_t,RC_t)}\right)
=
-\theta_1 x_t + \theta_2 RC_t
+ \beta\Bigl[
V(\min(x_t+1,7),RC_{t+1}) - V(1,RC_{t+1})
\Bigr].
$$

Now apply the Type-I-EV identity to the continuation states at $t+1$:

$$
V(x,RC_{t+1})
=
-\ln P(a=1\mid x,RC_{t+1}) + \delta_1(x,RC_{t+1}) + \gamma.
$$

Because $\delta_1(x,RC_{t+1})$ is the same for all $x$ once we condition on $RC_{t+1}$, it cancels in the difference, yielding

$$
V(\min(x_t+1,7),RC_{t+1}) - V(1,RC_{t+1})
=
-\ln P(a=1\mid \min(x_t+1,7),RC_{t+1})
+ \ln P(a=1\mid 1,RC_{t+1}).
$$

Substitute this into the log-odds equation:

$$
\ln\!\left(\frac{P(a=0\mid x_t,RC_t)}{P(a=1\mid x_t,RC_t)}\right)
+ \beta \ln\!\left(
\frac{P(a=1\mid \min(x_t+1,7),RC_{t+1})}
{P(a=1\mid 1,RC_{t+1})}
\right)
=
-\theta_1 x_t + \theta_2 RC_t.
$$

This is the Scott-style linear regression equation.

In our implementation, since $RC_t$ is common in each month, we write the empirical version as

$$
Y_{t,x}
=
\ln\!\left(\frac{\hat P_t(a=0\mid x)}{\hat P_t(a=1\mid x)}\right)
+ \beta \ln\!\left(
\frac{\hat P_{t+1}(a=1\mid \min(x+1,7))}
{\hat P_{t+1}(a=1\mid 1)}
\right)
=
-\theta_1 x + \theta_2 RC_t + u_{t,x}.
$$


In [55]:
log_odds_tx = np.log(P0_hat_tx) - np.log(P1_hat_tx)   # shape (100, 7)
log_p1_tx = np.log(P1_hat_tx)                         # shape (100, 7)

# Build state-level regression rows:
# one row for each (t, x) with t = 1,...,99
rows = []
for t_idx in range(T_MAX - 1):   # 0..98  <=>  t=1..99
    for x_idx in range(N_X):     # 0..6   <=>  x=1..7
        n_tx = N_tx[t_idx, x_idx]
        if n_tx <= 0:
            continue

        x = x_idx + 1
        x_next = min(x + 1, N_X)
        rc_t = rc_by_t[t_idx]

        y_tx = (
            log_odds_tx[t_idx, x_idx]
            + BETA * (log_p1_tx[t_idx + 1, x_next - 1] - log_p1_tx[t_idx + 1, 0])
        )

        rows.append((t_idx + 1, x, rc_t, n_tx, y_tx))

reg_df = pd.DataFrame(rows, columns=["t", "x", "rc", "weight", "y"])

print("Regression dataset head:")
print(reg_df.head(10))
print(f"\nNumber of state-level rows: {len(reg_df):,}")
print(f"Total weight (agent-period observations used): {reg_df['weight'].sum():,.0f}")


Regression dataset head:
   t  x    rc  weight         y
0  1  1  45.0   235.0  7.817618
1  1  2  45.0   168.0  7.203823
2  1  3  45.0   194.0  6.673091
3  1  4  45.0   114.0  6.254182
4  1  5  45.0   141.0  5.964445
5  1  6  45.0    71.0  5.765869
6  1  7  45.0    77.0  5.230818
7  2  1  49.0   156.0  8.350329
8  2  2  49.0   234.0  7.755339
9  2  3  49.0   166.0  7.425475

Number of state-level rows: 693
Total weight (agent-period observations used): 99,000


### Estimate $(\theta_1,\theta_2)$ by Weighted OLS

The regression equation is

$$
Y_{t,x} = -\theta_1 x + \theta_2 RC_t + u_{t,x}.
$$

Write this in matrix form:

$$
\mathbf{y} = X\beta + u,
\qquad
\beta =
\begin{bmatrix}
b_x \\
b_{RC}
\end{bmatrix}
=
\begin{bmatrix}
-\theta_1 \\
\theta_2
\end{bmatrix}.
$$

We use state-level rows indexed by $(t,x)$, but weight each row by the number of observations in that cell:

$$
W = \mathrm{diag}(n_{t,x}).
$$

Then weighted OLS is

$$
\hat\beta = (X^\top W X)^{-1} X^\top W \mathbf{y}.
$$

This is equivalent to observation-level OLS with one-agent-period-one-vote, because repeating the same state-level row $n_{t,x}$ times gives the same normal equations.


In [56]:
# Weighted OLS with no intercept:
# y = b_x * x + b_rc * rc + error
# where theta1 = -b_x and theta2 = b_rc

t0_theta = time.perf_counter()
y = reg_df["y"].to_numpy()
X = np.column_stack([
    reg_df["x"].to_numpy(),
    reg_df["rc"].to_numpy(),
])  # shape (n_rows, 2)
w = reg_df["weight"].to_numpy()

XtWX = X.T @ (w[:, None] * X)
XtWy = X.T @ (w * y)
beta_hat = np.linalg.solve(XtWX, XtWy)

b_x_hat, b_rc_hat = beta_hat
theta1_hat = -b_x_hat
theta2_hat = b_rc_hat

# Simple frequency-weighted OLS variance formula
resid = y - X @ beta_hat
n_obs_equiv = int(w.sum())
k = X.shape[1]
sigma2_hat = (w * resid ** 2).sum() / (n_obs_equiv - k)
vcov_hat = sigma2_hat * np.linalg.inv(XtWX)
se_beta = np.sqrt(np.diag(vcov_hat))
se_theta1 = se_beta[0]
se_theta2 = se_beta[1]

# Weighted R^2
y_bar_w = np.average(y, weights=w)
ssr = np.sum(w * resid ** 2)
sst = np.sum(w * (y - y_bar_w) ** 2)
r2_w = 1.0 - ssr / sst
theta_runtime_sec = time.perf_counter() - t0_theta
theta_runtime_ms = 1000.0 * theta_runtime_sec
running_time_sec = time.perf_counter() - t0_method
running_time_ms = 1000.0 * running_time_sec

print("Weighted OLS coefficients [b_x, b_rc]:")
print(np.round(beta_hat, 6))
print("\nImplied theta estimates:")
print(f"theta1_hat = {-b_x_hat:.6f}")
print(f"theta2_hat = {b_rc_hat:.6f}")
print("\nApproximate standard errors:")
print(f"se(theta1_hat) = {se_theta1:.6f}")
print(f"se(theta2_hat) = {se_theta2:.6f}")
print(f"Weighted R^2    = {r2_w:.6f}")
print(f"OLS-only time   = {theta_runtime_sec:.4f} seconds ({theta_runtime_ms:.2f} ms)")
print(f"Comparable running time = {running_time_sec:.4f} seconds ({running_time_ms:.2f} ms)")


Weighted OLS coefficients [b_x, b_rc]:
[-0.473937  0.172606]

Implied theta estimates:
theta1_hat = 0.473937
theta2_hat = 0.172606

Approximate standard errors:
se(theta1_hat) = 0.002443
se(theta2_hat) = 0.000210
Weighted R^2    = 0.459507
OLS-only time   = 0.0010 seconds (1.04 ms)
Comparable running time = 0.1185 seconds (118.55 ms)


### Weighting Interpretation

Here we use state-level weighted OLS, with one row per $(t,x)$ cell and weight equal to the number of bus-month observations in that cell.

So the estimator is numerically equivalent to observation-level OLS with one-agent-period-one-vote:

- cells with more observations get more weight,
- rare $(t,x)$ cells get less weight,
- this matches the implicit weighting in Parts (a), (b), and (c), where each observation contributes to the criterion.


In [57]:
# Verify equivalence with observation-level OLS:
# merge the state-level y_{t,x} back to each observation and run plain OLS
obs_df = df.merge(reg_df[["t", "x", "y"]], on=["t", "x"], how="left")
obs_df = obs_df[obs_df["t"] <= T_MAX - 1].copy()   # only t=1,...,99 have continuation terms

X_obs = np.column_stack([
    obs_df["x"].to_numpy(),
    obs_df["rc"].to_numpy(),
])
y_obs = obs_df["y"].to_numpy()

beta_hat_obs = np.linalg.solve(X_obs.T @ X_obs, X_obs.T @ y_obs)

print("Observation-level OLS coefficients [b_x, b_rc]:")
print(np.round(beta_hat_obs, 6))
print("\nDifference from weighted state-level OLS:")
print(np.round(beta_hat_obs - beta_hat, 12))


Observation-level OLS coefficients [b_x, b_rc]:
[-0.473937  0.172606]

Difference from weighted state-level OLS:
[ 0. -0.]


### Results


In [58]:
print("=" * 62)
print("        PART (d) -- SCOTT (2014) LINEAR REGRESSION")
print("=" * 62)
print(f"theta1_hat    (mileage cost coeff.)     = {theta1_hat:.4f}")
print(f"theta2_hat    (replacement cost coeff.) = {theta2_hat:.4f}")
print("-" * 62)
print(f"Weighted R^2                              = {r2_w:.4f}")
print(f"State-level rows used                     = {len(reg_df):,}")
print(f"Observation-equivalent sample size        = {n_obs_equiv:,}")
print(f"OLS-only time                             = {theta_runtime_sec:.4f} sec ({theta_runtime_ms:.2f} ms)")
print(f"Comparable running time                   = {running_time_sec:.4f} sec ({running_time_ms:.2f} ms)")
print("-" * 62)
print("Interpretation:")
print("  coefficient on x   = -theta1")
print("  coefficient on RC  =  theta2")
print("=" * 62)


        PART (d) -- SCOTT (2014) LINEAR REGRESSION
theta1_hat    (mileage cost coeff.)     = 0.4739
theta2_hat    (replacement cost coeff.) = 0.1726
--------------------------------------------------------------
Weighted R^2                              = 0.4595
State-level rows used                     = 693
Observation-equivalent sample size        = 99,000
OLS-only time                             = 0.0010 sec (1.04 ms)
Comparable running time                   = 0.1185 sec (118.55 ms)
--------------------------------------------------------------
Interpretation:
  coefficient on x   = -theta1
  coefficient on RC  =  theta2


In [59]:
master_results['Linear Regression Approach'] = {
    'part': '(d)',
    'method': 'Linear Regression Approach',
    'theta1': float(theta1_hat),
    'theta2': float(theta2_hat),
    'running_time_sec': float(running_time_sec),
    'running_time_ms': float(running_time_ms),
}
print('Saved summary for Part (d) in master_results.')


Saved summary for Part (d) in master_results.


### Summary

- We do not solve a Bellman equation.
- We do not invert a Hotz-Miller linear system.
- We do not simulate forward to approximate the value function.
- We do not estimate the RC transition process explicitly.

Instead, we use the special structure of the replacement action:

- replacing resets mileage to 1,
- so the continuation value difference can be rewritten using observable CCP differences,
- which turns the model into a linear regression in $(\theta_1,\theta_2)$.

**Note:** This estimator depends heavily on the quality of the CCP estimates:
- if some $(t,x)$ cells are sparse,
- or if the estimated CCPs are noisy,
- then the log terms can become unstable.

That is why we used a smooth cubic-logit CCP estimator rather than raw unsmoothed frequencies.



## Final Comparison Table

The next cell collects the results saved from Parts (a), (b), (c.1), (c.2), and (d),
builds the methodology-comparison table, and exports LaTeX versions of the table.


In [60]:

from pathlib import Path
from IPython.display import Markdown, display

comparison_order = [
    "Nested Fixed Point",
    "Hotz and Miller",
    "Forward Simulation (Eq. 5)",
    "Forward Simulation (Eq. 6)-(7)",
    "Linear Regression Approach",
]

records = [master_results[name] for name in comparison_order]
summary_df = pd.DataFrame(records)

display(Markdown("### Compact Comparison Table"))
display(summary_df[["part", "method", "theta1", "theta2", "running_time_sec", "running_time_ms"]])

report_rows = []
for row in records:
    report_rows.extend(
        [
            {"Method": row["method"], "Parameter": "theta1", "Paola": f"{row['theta1']:.4f}"},
            {"Method": row["method"], "Parameter": "theta2", "Paola": f"{row['theta2']:.4f}"},
            {
                "Method": row["method"],
                "Parameter": "Running Time",
                "Paola": f"{row['running_time_sec']:.4f} s ({row['running_time_ms']:.2f} ms)",
            },
        ]
    )

report_df = pd.DataFrame(report_rows)
compact_display_df = summary_df.copy()
compact_display_df["Running Time"] = compact_display_df.apply(
    lambda row: f"{row['running_time_sec']:.4f} s ({row['running_time_ms']:.2f} ms)",
    axis=1,
)

display(Markdown("### Report-Style Table"))
display(report_df)

compact_latex = compact_display_df[["part", "method", "theta1", "theta2", "Running Time"]].rename(
    columns={"part": "Part", "method": "Method", "theta1": "theta1", "theta2": "theta2"}
).to_latex(
    index=False,
    escape=False,
    caption="Results obtained for Rust's Bus Problem using five estimation methods",
    label="tab:rust_bus_paola_compact",
)

report_latex = report_df.to_latex(
    index=False,
    escape=False,
    caption="Results obtained for Rust's Bus Problem using five estimation methods",
    label="tab:rust_bus_paola",
)

compact_path = Path.cwd() / "ECON662_HW2_pv_results_table_compact.tex"
report_path = Path.cwd() / "ECON662_HW2_pv_results_table.tex"
compact_path.write_text(compact_latex)
report_path.write_text(report_latex)

print("Wrote compact LaTeX table to:", compact_path.name)
print("Wrote report-style LaTeX table to:", report_path.name)

if "Nested Fixed Point" in master_common:
    rho = master_common["Nested Fixed Point"]
    print(
        "Common first-stage OLS estimates used in the dynamic-MLE sections: "
        f"rho0={rho['rho0']:.4f}, rho1={rho['rho1']:.4f}, sigma_rho={rho['sigma_rho']:.4f}"
    )

print("\nCompact LaTeX table:\n")
print(compact_latex)

print("\nReport-style LaTeX table:\n")
print(report_latex)


### Compact Comparison Table

,part,method,theta1,theta2,running_time_sec,running_time_ms
0,(a),Nested Fixed Point,0.408839,0.156278,0.821219,821.218833
1,(b),Hotz and Miller,0.408785,0.156299,0.008055,8.055292
2,(c.1),Forward Simulation (Eq. 5),0.395534,0.153706,0.193344,193.343628
3,(c.2),Forward Simulation (Eq. 6)-(7),0.389652,0.153694,0.270279,270.278636
4,(d),Linear Regression Approach,0.473937,0.172606,0.118549,118.548666


### Report-Style Table

,Method,Parameter,Paola
0,Nested Fixed Point,theta1,0.4088
1,Nested Fixed Point,theta2,0.1563
2,Nested Fixed Point,Running Time,0.8212 s (821.22 ms)
3,Hotz and Miller,theta1,0.4088
4,Hotz and Miller,theta2,0.1563
5,Hotz and Miller,Running Time,0.0081 s (8.06 ms)
6,Forward Simulation (Eq. 5),theta1,0.3955
7,Forward Simulation (Eq. 5),theta2,0.1537
8,Forward Simulation (Eq. 5),Running Time,0.1933 s (193.34 ms)
9,Forward Simulation (Eq. 6)-(7),theta1,0.3897


Wrote compact LaTeX table to: ECON662_HW2_pv_results_table_compact.tex
Wrote report-style LaTeX table to: ECON662_HW2_pv_results_table.tex
Common first-stage OLS estimates used in the dynamic-MLE sections: rho0=17.8269, rho1=0.6249, sigma_rho=4.6060

Compact LaTeX table:

\begin{table}
\caption{Results obtained for Rust's Bus Problem using five estimation methods}
\label{tab:rust_bus_paola_compact}
\begin{tabular}{llrrl}
\toprule
Part & Method & theta1 & theta2 & Running Time \\
\midrule
(a) & Nested Fixed Point & 0.408839 & 0.156278 & 0.8212 s (821.22 ms) \\
(b) & Hotz and Miller & 0.408785 & 0.156299 & 0.0081 s (8.06 ms) \\
(c.1) & Forward Simulation (Eq. 5) & 0.395534 & 0.153706 & 0.1933 s (193.34 ms) \\
(c.2) & Forward Simulation (Eq. 6)-(7) & 0.389652 & 0.153694 & 0.2703 s (270.28 ms) \\
(d) & Linear Regression Approach & 0.473937 & 0.172606 & 0.1185 s (118.55 ms) \\
\bottomrule
\end{tabular}
\end{table}


Report-style LaTeX table:

\begin{table}
\caption{Results obtained for Rust